# Controlled Unlearning Prompt A/B Test Pipeline — Provider-Separated API Fixes

This notebook preserves the corrected prompt parity/codebook-example design from v3, but fixes three API and result-capture problems found during live execution:

1. **Claude Sonnet 5:** does not receive the deprecated `temperature` parameter; the Python SDK `messages.parse()` helper is used with Pydantic structured output and explicit effort.
2. **Gemini 2.5 Flash:** raw JSON Schema is sent through `response_json_schema`, not the older typed `response_schema` path that converted `additionalProperties` into an invalid payload field.
3. **Confidence:** `confidence` is a required 0–1 field in the shared schema and is parsed/saved for all three providers.

The three providers now run in **separate notebook sections**, with separate JSONL resume logs and separate prediction CSV snapshots. A provider stops after its first final API error by default so a configuration bug does not create dozens of duplicate failures.


In [ ]:
# Optional: install/update dependencies once. Uncomment if needed.
%pip install -q -U pandas openpyxl tqdm python-dotenv tiktoken openai anthropic google-genai pydantic


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 73.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 676.6/676.6 kB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 52.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 956.9/956.9 kB 40.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.


## 1. Configuration

Set the workbook path and provider-specific run switches here. The provider sections later in the notebook are independent: run the OpenAI cell, inspect it, then Claude, then Gemini.

`STOP_PROVIDER_ON_FIRST_ERROR = True` is deliberate for diagnosis. A malformed request should fail once, save the error, and stop that provider rather than retrying the same bad configuration across every paragraph.


In [ ]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

In [ ]:
from pathlib import Path
from datetime import datetime, timezone, timedelta

from zoneinfo import ZoneInfo
import os
import re
import json
import time
import math
import hashlib
from collections import defaultdict
from typing import Literal

import pandas as pd
import numpy as np
from IPython.display import display
from pydantic import BaseModel, ConfigDict, Field, ValidationError

try:
    from dotenv import load_dotenv
except ImportError:
    def load_dotenv(*args, **kwargs):
        return False

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(iterable=None, total=None, desc=None, **kwargs):
        return iterable if iterable is not None else range(total or 0)

load_dotenv(dotenv_path=Path(".env"))

# Workbook location.
WORKBOOK_PATH = Path("/content/Unlearning Codebook.xlsx")
CODEBOOK_SHEET = "Codebook"

OUTPUT_DIR = Path("outputs_unlearning_ab_test_v4")
OUTPUT_DIR.mkdir(exist_ok=True)

# Prompt/run version is part of every resume key. Change it whenever prompt/schema logic changes.
PROMPT_VERSION = "v4_provider_separated_confidence_api_fixes"

# Provider-specific execution switches. Each provider has its own run cell later.
RUN_OPENAI_CALLS = True
RUN_ANTHROPIC_CALLS = True
RUN_GEMINI_CALLS = True

# Stop a provider after its first final API/configuration error. Recommended while diagnosing.
STOP_PROVIDER_ON_FIRST_ERROR = True

# Use a small row limit while testing. Set to None for the full GPT Test sheet.
ROW_LIMIT = None

# Use all labeled examples by default. For budget tests, try 8, 12, or 20.
MAX_FEWSHOT_EXAMPLES = None

# Optional paragraph truncation. None preserves the complete paragraph.
MAX_TEXT_CHARS = None

# Output/API settings.
MAX_OUTPUT_TOKENS = 2048
REQUEST_SLEEP_SECONDS = 0.2
RESUME_FROM_JSONL = True

# Hard preflight budget guard PER PROVIDER.
MAX_ESTIMATED_COST_USD_PER_PROVIDER = 5.00

# USD per 1M tokens. Pricing snapshot used for the experiment; update if provider pricing changes.
MODEL_CONFIG = [
    {
        "provider": "openai",
        "model": "gpt-5.4",
        "enabled": True,
        "api_key_env": "OPENAI_API_KEY",
        "input_usd_per_1m": 0.20,
        "output_usd_per_1m": 1.25,
        "reasoning_effort": "low",
        "request_sleep_seconds": 0.2,
    },
    {
        "provider": "anthropic",
        "model": "claude-haiku-4-5-20251001",
        "enabled": True,
        "api_key_env": "ANTHROPIC_API_KEY",
        "input_usd_per_1m": 1.00,
        "output_usd_per_1m": 5.00,
#        "effort": "low",
        "request_sleep_seconds": 0.2,
    },
    {
        "provider": "gemini",
        "model": "gemini-3.5-flash",
        "enabled": True,
        "api_key_env": "GEMINI_API_KEY",
        "input_usd_per_1m": 0.75,
        "output_usd_per_1m": 4.5,
        "thinking_level": "low",
        "request_sleep_seconds": 0.2,
        },
]

PROMPT_VARIANTS_TO_RUN = [
    "direct_no_context",
    "definitions_only",
    "definitions_examples_no_metadata",
    "definitions_examples_with_metadata",
]

print("Workbook path:", WORKBOOK_PATH)
print("Codebook sheet:", CODEBOOK_SHEET)
print("Prompt version:", PROMPT_VERSION)
print("Output folder:", OUTPUT_DIR.resolve())
print("Provider run switches:", {
    "openai": RUN_OPENAI_CALLS,
    "anthropic": RUN_ANTHROPIC_CALLS,
    "gemini": RUN_GEMINI_CALLS,
})


Workbook path: /content/Unlearning Codebook.xlsx
Codebook sheet: Codebook
Prompt version: v4_provider_separated_confidence_api_fixes
Output folder: /content/outputs_unlearning_ab_test_v4
Provider run switches: {'openai': True, 'anthropic': True, 'gemini': True}


## 2. Load workbook sheets

The workbook is expected to contain these sheets:

- `Examples`
- `GPT Test`
- `Codebook`
- `Decision Rules`

In [ ]:
def drop_empty_unnamed_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df = df.dropna(axis=1, how="all")
    df = df.loc[:, ~df.columns.astype(str).str.match(r"^Unnamed", na=False)]
    return df


examples_df = drop_empty_unnamed_columns(pd.read_excel(WORKBOOK_PATH, sheet_name="Examples"))
test_df = drop_empty_unnamed_columns(pd.read_excel(WORKBOOK_PATH, sheet_name="GPT Test"))
raw_codebook_df = pd.read_excel(WORKBOOK_PATH, sheet_name=CODEBOOK_SHEET, header=None)
decision_rules_df = drop_empty_unnamed_columns(pd.read_excel(WORKBOOK_PATH, sheet_name="Decision Rules"))

# Extract the objective and codebook table.
# The first row may contain the objective; the real header row starts where column A == "Code".
objective_text = str(raw_codebook_df.iloc[0, 0]) if not pd.isna(raw_codebook_df.iloc[0, 0]) else ""
header_candidates = raw_codebook_df.index[
    raw_codebook_df.iloc[:, 0].astype(str).str.strip().str.lower().eq("code")
].tolist()
if not header_candidates:
    raise ValueError(
        f"Could not find the codebook table header row in sheet {CODEBOOK_SHEET!r}; "
        "expected a row where column A equals 'Code'."
    )

codebook_header_row = header_candidates[0]
codebook_df = raw_codebook_df.iloc[codebook_header_row + 1:].copy()
codebook_df.columns = [
    str(x).strip() if not pd.isna(x) else ""
    for x in raw_codebook_df.iloc[codebook_header_row].tolist()
]
codebook_df = codebook_df.dropna(axis=0, how="all")
codebook_df = codebook_df.loc[:, [c for c in codebook_df.columns if c != ""]]
codebook_df = codebook_df.reset_index(drop=True)

required_codebook_cols = {
    "Code", "Definition", "Detection Logic", "Examples",
    "Positive Clarification", "Negative Clarification",
}
missing_codebook_cols = required_codebook_cols - set(codebook_df.columns)
if missing_codebook_cols:
    raise ValueError(
        "The controlled prompt requires these codebook columns, but they are missing: "
        + ", ".join(sorted(missing_codebook_cols))
    )

print("Examples shape:", examples_df.shape)
print("GPT Test shape:", test_df.shape)
print("Codebook table shape:", codebook_df.shape)
print("Decision Rules shape:", decision_rules_df.shape)
print("Codebook columns:", list(codebook_df.columns))

display(examples_df.head(3))
display(test_df.head(3))
display(codebook_df.head(8))


Examples shape: (62, 7)
GPT Test shape: (11, 8)
Codebook table shape: (13, 6)
Decision Rules shape: (4, 7)
Codebook columns: ['Code', 'Definition', 'Detection Logic', 'Examples', 'Positive Clarification', 'Negative Clarification']


,Number,Reference,Text Content,Rationale,Source,Unlearning,Codes
0,1:1,p 3,"Coordination within EPA, with State and local ...",Not unlearning. As no logic being abandoned. o...,EPA,No,"Depth: Moderate depth, Nature: Technical Natur..."
1,1:2,pp 9 – 10,"EPA officials told us that, in some instances,...",Not unlearning. As no logic being abandoned ju...,EPA,No,"Nature: Technical Nature, Pillar: Merging (Int..."
2,1:3,p 11,"From our review of wastewater issues, we found...",Unlearning as it states EPA was not commuunica...,EPA,Yes,"Depth: Low depth, Nature: Technical Nature, Pi..."


,Number,Text Content,Document,Codes,Unlearning,Pillar,Depth,Nature
0,14:2,Because of FEMA’s mission performance during H...,gao-06-442t.pdf,"Depth: High depth, Nature: Technical Nature, P...",Yes,Discarding,High,Technical
1,14:3,"In addition, we have done a great deal of work...",gao-06-442t.pdf,"Depth: High depth, Nature: Technical Nature, P...",Yes,Discarding,High,Technical
2,14:4,Because of FEMA’s mission performance during H...,gao-06-442t.pdf,"Depth: High depth, Pillar: Discarding (Normati...",Yes,Discarding,High,NaN


,Code,Definition,Detection Logic,Examples,Positive Clarification,Negative Clarification
0,Secondary Data from US govt agencies evaluatio...,"Text drawn from formal evaluations, audits, or...","The ""Subtractive"" Test",Passages that speak to unlearning framework,Step 1: Identify if Unlearning exists.\nStep 2...,NaN
1,Binary classification based on unlearning defi...,NaN,NaN,NaN,NaN,NaN
2,Unlearning,Unlearning refers to the deliberate process by...,Does this text advocate for change that implie...,"As the White House report states, “Ultimately,...",Code when the text explicitly calls for:\nAban...,"""Do NOT code if:\nThe text only discusses “les..."
3,Unlearning typology (Primary Analytic Codes),NaN,NaN,NaN,NaN,NaN
4,Reconsidering (Epistemic Unlearning),This involves the unlearning of dominant knowl...,Are they questioning the truth or authority of...,"""The question of how and when an event becomes...",Code when the text:\nQuestions how presidentia...,Do NOT code if:\nThe text proposes a technical...
5,Discarding (Normative Unlearning),This entails the rejection and abandonment of ...,Are they terminating a policy or law or organi...,"""removing statutory restrictions on DOD’s auth...",Code when the text:\n\nCalls for eliminating s...,"Do NOT code if:\nThe text suggests reform, adj..."
6,Realignment (Technical Unlearning),This pillar addresses the obsolescence or over...,Are they admitting an engineering or technical...,"""Andrew and Hugo, we identified the need for t...",Code when the text:\nCalls for revising engine...,Do not code if it is a routine maintenance upd...
7,Merging (Integrative Unlearning),This pillar rests on overcoming disciplinary a...,Are they destroying silos to integrate knowledge?,"""Such operational plans should, for example, f...",Code when the text:\nExplicitly calls for join...,Do NOT code if:\nCollaboration is mentioned sy...


## 3. Preprocess labels and expand the `Codes` column

This section parses values such as:

`Depth: Moderate depth, Nature: Technical Nature, Pillar: Merging (Integrative), Stakeholder: EPA, Unlearning`

into structured columns like:

- `code_depth`
- `code_nature`
- `code_pillar`
- `code_stakeholder`
- `code_unlearning`

It also creates optional one-hot indicator columns such as `code_pillar__merging_integrative`.

In [ ]:
def normalize_yes_no(value):
    """Normalize yes/no-like values to 'Yes', 'No', or np.nan."""
    if pd.isna(value):
        return np.nan
    s = str(value).strip().lower()
    if s in {"yes", "y", "true", "1", "unlearning"}:
        return "Yes"
    if s in {"no", "n", "false", "0", "not unlearning", "non-unlearning"}:
        return "No"
    if "yes" == s[:3]:
        return "Yes"
    if "no" == s[:2]:
        return "No"
    return np.nan


def clean_text_for_prompt(text, max_chars=MAX_TEXT_CHARS):
    if pd.isna(text):
        return ""
    s = re.sub(r"\s+", " ", str(text)).strip()
    if max_chars is not None and len(s) > max_chars:
        s = s[:max_chars].rstrip() + " ... [TRUNCATED]"
    return s


def slugify(value):
    s = str(value).strip().lower()
    s = re.sub(r"[^a-z0-9]+", "_", s)
    return s.strip("_") or "blank"


def parse_codes_cell(cell):
    """Parse comma-separated code metadata into a dict of lists."""
    parsed = defaultdict(list)
    if pd.isna(cell):
        return parsed
    text = str(cell).strip()
    if not text:
        return parsed

    parts = [p.strip() for p in text.split(",") if p.strip()]
    for part in parts:
        if ":" in part:
            key, value = part.split(":", 1)
            key = slugify(key)
            value = value.strip()
            if value and value not in parsed[key]:
                parsed[key].append(value)
        else:
            # Handles standalone values such as "Unlearning"
            key = slugify(part)
            if key == "unlearning":
                parsed["unlearning"].append("Yes")
            else:
                parsed["flag"].append(part)
    return parsed


def expand_codes_column(df: pd.DataFrame, code_col: str = "Codes", add_indicators: bool = True) -> pd.DataFrame:
    df = df.copy()
    if code_col not in df.columns:
        return df

    parsed_rows = [parse_codes_cell(v) for v in df[code_col]]
    all_keys = sorted({k for row in parsed_rows for k in row.keys()})

    for key in all_keys:
        df[f"code_{key}"] = ["; ".join(row.get(key, [])) for row in parsed_rows]

    if add_indicators:
        indicator_values = sorted({
            (key, value)
            for row in parsed_rows
            for key, values in row.items()
            for value in values
        })
        for key, value in indicator_values:
            col = f"code_{key}__{slugify(value)}"
            df[col] = [int(value in row.get(key, [])) for row in parsed_rows]

    return df


examples_df = expand_codes_column(examples_df, "Codes", add_indicators=True)
test_df = expand_codes_column(test_df, "Codes", add_indicators=True)

if "Unlearning" not in test_df.columns:
    raise ValueError("GPT Test sheet must contain a ground-truth 'Unlearning' column.")
if "Text Content" not in test_df.columns:
    raise ValueError("GPT Test sheet must contain a 'Text Content' column.")
if "Text Content" not in examples_df.columns or "Unlearning" not in examples_df.columns:
    raise ValueError("Examples sheet must contain 'Text Content' and 'Unlearning' columns.")

test_df["ground_truth_unlearning"] = test_df["Unlearning"].apply(normalize_yes_no)
examples_df["example_unlearning"] = examples_df["Unlearning"].apply(normalize_yes_no)

test_df = test_df.dropna(subset=["Text Content", "ground_truth_unlearning"]).reset_index(drop=True)
examples_df = examples_df.dropna(subset=["Text Content", "example_unlearning"]).reset_index(drop=True)

if ROW_LIMIT is not None:
    test_df = test_df.head(ROW_LIMIT).copy()

print("Clean examples:", examples_df.shape)
print("Clean test rows:", test_df.shape)
print("Expanded code columns in GPT Test:")
print([c for c in test_df.columns if c.startswith("code_")][:30])

preview_cols = ["Number", "ground_truth_unlearning"] + [
    c for c in ["code_depth", "code_nature", "code_pillar"] if c in test_df.columns
]
display(test_df[preview_cols].head())

Clean examples: (6, 30)
Clean test rows: (11, 26)
Expanded code columns in GPT Test:
['code_depth', 'code_nature', 'code_pillar', 'code_stakeholder', 'code_unlearning', 'code_depth__high_depth', 'code_depth__moderate_depth', 'code_nature__normative_nature', 'code_nature__technical_nature', 'code_pillar__discarding_normative', 'code_pillar__merging_integrative', 'code_pillar__reconsidering_epistemic', 'code_stakeholder__congress', 'code_stakeholder__dept_of_defense', 'code_stakeholder__fema', 'code_stakeholder__miscelleanous', 'code_unlearning__yes']


,Number,ground_truth_unlearning,code_depth,code_nature,code_pillar
0,14:2,Yes,High depth,Technical Nature,Discarding (Normative)
1,14:3,Yes,High depth,Technical Nature,Discarding (Normative)
2,14:4,Yes,High depth,,Discarding (Normative)
3,14:5,Yes,Moderate depth,Normative Nature,Reconsidering (Epistemic)
4,14:6,Yes,Moderate depth,Normative Nature,Reconsidering (Epistemic)


## 4. Build reusable prompt context

The codebook serializer intentionally includes **Code, Definition, Detection Logic, Examples, Positive Clarification, and Negative Clarification**. This fixes the prior omission of the codebook's own examples.

For human-labeled examples:

- `definitions_examples_no_metadata` includes paragraph text plus human labels (`Unlearning` and `Codes`).
- `definitions_examples_with_metadata` adds available provenance/coder context such as reference, rationale, and source.


In [ ]:
def choose_examples(
    df: pd.DataFrame,
    max_examples=MAX_FEWSHOT_EXAMPLES,
    seed: int = 42,
) -> pd.DataFrame:
    """Use all examples by default; otherwise sample approximately label-balanced examples."""
    if max_examples is None or max_examples >= len(df):
        return df.copy()

    groups = []
    labels = ["Yes", "No"]
    per_label = max(1, max_examples // len(labels))

    for label in labels:
        sub = df[df["example_unlearning"] == label]
        if len(sub) == 0:
            continue
        take = min(per_label, len(sub))
        groups.append(sub.sample(n=take, random_state=seed))

    selected = pd.concat(groups, ignore_index=False) if groups else df.head(0)

    remaining = max_examples - len(selected)
    if remaining > 0:
        rest = df.drop(index=selected.index, errors="ignore")
        if len(rest) > 0:
            selected = pd.concat([
                selected,
                rest.sample(n=min(remaining, len(rest)), random_state=seed),
            ])

    return selected.sort_index().reset_index(drop=True)


prompt_examples_df = choose_examples(examples_df, MAX_FEWSHOT_EXAMPLES)
print(f"Using {len(prompt_examples_df)} human-labeled examples in few-shot prompts.")


def _prompt_value(value) -> str:
    if pd.isna(value):
        return ""
    return clean_text_for_prompt(value, max_chars=None)


def make_codebook_text(codebook: pd.DataFrame) -> str:
    """
    Serialize the SAME codebook fields for every definitions-based prompt.

    The original nano notebook accidentally omitted the codebook's own Examples column.
    This version preserves it explicitly.
    """
    parts = []
    for _, row in codebook.iterrows():
        code_name = _prompt_value(row.get("Code", ""))
        if not code_name or code_name.lower() == "nan":
            continue

        block = [
            f"Code: {code_name}",
            f"Definition: {_prompt_value(row.get('Definition', ''))}",
            f"Detection logic: {_prompt_value(row.get('Detection Logic', ''))}",
            f"Examples: {_prompt_value(row.get('Examples', ''))}",
            f"Positive: {_prompt_value(row.get('Positive Clarification', ''))}",
            f"Negative: {_prompt_value(row.get('Negative Clarification', ''))}",
        ]
        parts.append("\n".join(block))

    return "\n\n".join(parts)


def make_examples_text(examples: pd.DataFrame, include_metadata: bool) -> str:
    """
    No-metadata examples contain paragraph text + HUMAN LABELS.
    Metadata examples add provenance/coder metadata but do not duplicate expanded one-hot code columns.
    """
    blocks = []

    for i, (_, row) in enumerate(examples.iterrows(), start=1):
        text = clean_text_for_prompt(row.get("Text Content", ""), max_chars=MAX_TEXT_CHARS)
        label = normalize_yes_no(row.get("Unlearning", row.get("example_unlearning", "")))

        block = [
            f"Example {i}",
            "Text Content:",
            text,
            "Human labels:",
            f"Unlearning: {label}",
        ]

        codes = row.get("Codes", "")
        if not pd.isna(codes) and str(codes).strip():
            block.append(f"Codes: {_prompt_value(codes)}")

        if include_metadata:
            for col in ["Number", "Reference", "Rationale", "Source"]:
                if col in row.index:
                    val = row.get(col, "")
                    if not pd.isna(val) and str(val).strip():
                        block.append(f"{col}: {_prompt_value(val)}")

        blocks.append("\n".join(block))

    return "\n\n---\n\n".join(blocks)


CODEBOOK_TEXT = make_codebook_text(codebook_df)
EXAMPLES_TEXT_NO_METADATA = make_examples_text(prompt_examples_df, include_metadata=False)
EXAMPLES_TEXT_WITH_METADATA = make_examples_text(prompt_examples_df, include_metadata=True)

print("Codebook context characters:", len(CODEBOOK_TEXT))
print("Examples without metadata characters:", len(EXAMPLES_TEXT_NO_METADATA))
print("Examples with metadata characters:", len(EXAMPLES_TEXT_WITH_METADATA))


Using 6 human-labeled examples in few-shot prompts.
Codebook context characters: 10680
Examples without metadata characters: 6457
Examples with metadata characters: 7158


## 5. Prompt variants and parity audit

The core task wording, JSON schema note, system message, paragraph-ID label, and paragraph label mirror the GPT-4o-mini notebook.

The only intended differences among variants are:

1. `direct_no_context`: no codebook and no human-labeled examples.
2. `definitions_only`: codebook only, including the codebook's own `Examples` field.
3. `definitions_examples_no_metadata`: codebook + paragraph text + human labels.
4. `definitions_examples_with_metadata`: codebook + human-labeled examples + available example metadata.


In [ ]:
BASE_SYSTEM_PROMPT = "Return valid JSON only. Do not include markdown."

# This task block is kept exactly aligned with the GPT-4o-mini notebook.
COMMON_TASK_BLOCK = """You are coding federal disaster policy text using the unlearning codebook.

Task:
1. Decide whether the paragraph contains unlearning.
2. If yes, assign one target type.
3. Identify the agency or sub-unit if the paragraph states one.
4. Keep the rationale short.""".strip()

# Confidence is intentionally added to the output schema. The original 4o-mini schema had no confidence field,
# so exact schema parity and confidence capture are logically incompatible. The classification task block remains exact.
JSON_SCHEMA_NOTE = """
Return valid JSON only with this structure:
{
  "unlearning_present": true/false,
  "target_type": "Leadership | laws_plans_policies | capabilities | funds_resources | misc_organizational | none",
  "agency": "agency or sub-unit name if stated, otherwise null",
  "confidence": "number from 0.0 to 1.0",
  "rationale": "one short sentence"
}
""".strip()


class AnnotationOutput(BaseModel):
    model_config = ConfigDict(extra="forbid")

    unlearning_present: bool
    target_type: Literal[
        "Leadership",
        "laws_plans_policies",
        "capabilities",
        "funds_resources",
        "misc_organizational",
        "none",
    ]
    agency: str | None
    confidence: float = Field(
        ge=0.0,
        le=1.0,
        description="Model's confidence in the classification, from 0.0 to 1.0.",
    )
    rationale: str


ANNOTATION_JSON_SCHEMA = AnnotationOutput.model_json_schema()


def build_user_prompt(variant_name: str, test_row: pd.Series) -> str:
    """
    Render one controlled prompt.

    The classification task block and ordering match the GPT-4o-mini scaffold:
    task block -> optional codebook/examples -> schema -> Paragraph ID -> Paragraph.
    """
    if variant_name not in PROMPT_VARIANTS_TO_RUN:
        raise ValueError(f"Unknown prompt variant: {variant_name}")

    text = clean_text_for_prompt(test_row["Text Content"], max_chars=MAX_TEXT_CHARS)
    paragraph_id = test_row.get("Number", test_row.name)

    if variant_name == "direct_no_context":
        return f"""{COMMON_TASK_BLOCK}

{JSON_SCHEMA_NOTE}

Paragraph ID: {paragraph_id}
Paragraph:
{text}""".strip()

    if variant_name == "definitions_only":
        return f"""{COMMON_TASK_BLOCK}

Codebook:
{CODEBOOK_TEXT}

{JSON_SCHEMA_NOTE}

Paragraph ID: {paragraph_id}
Paragraph:
{text}""".strip()

    examples_text = (
        EXAMPLES_TEXT_NO_METADATA
        if variant_name == "definitions_examples_no_metadata"
        else EXAMPLES_TEXT_WITH_METADATA
    )

    return f"""{COMMON_TASK_BLOCK}

Codebook:
{CODEBOOK_TEXT}

Labeled examples:
{examples_text}

{JSON_SCHEMA_NOTE}

Paragraph ID: {paragraph_id}
Paragraph:
{text}""".strip()


def build_prompt_messages(variant_name: str, test_row: pd.Series) -> list[dict]:
    return [
        {"role": "system", "content": BASE_SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt(variant_name, test_row)},
    ]


def prompt_sha256(messages: list[dict]) -> str:
    canonical = json.dumps(messages, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(canonical.encode("utf-8")).hexdigest()


# ---- Prompt parity / contamination assertions ----
preview_row = test_df.iloc[0]

EXPECTED_4O_TASK_BLOCK = """You are coding federal disaster policy text using the unlearning codebook.

Task:
1. Decide whether the paragraph contains unlearning.
2. If yes, assign one target type.
3. Identify the agency or sub-unit if the paragraph states one.
4. Keep the rationale short.""".strip()
assert COMMON_TASK_BLOCK == EXPECTED_4O_TASK_BLOCK, "The classification task block drifted from GPT-4o-mini."

nonempty_codebook_examples = [
    _prompt_value(v)
    for v in codebook_df["Examples"]
    if not pd.isna(v) and str(v).strip()
]
assert "Examples:" in CODEBOOK_TEXT, "The codebook Examples field is missing from CODEBOOK_TEXT."
if nonempty_codebook_examples:
    assert nonempty_codebook_examples[0] in CODEBOOK_TEXT, "A codebook example was not preserved."

assert "Labeled examples:" not in build_user_prompt("definitions_only", preview_row)
assert "Labeled examples:" not in build_user_prompt("direct_no_context", preview_row)
assert "Labeled examples:" in build_user_prompt("definitions_examples_no_metadata", preview_row)
assert "Labeled examples:" in build_user_prompt("definitions_examples_with_metadata", preview_row)

no_meta_prompt = build_user_prompt("definitions_examples_no_metadata", preview_row)
with_meta_prompt = build_user_prompt("definitions_examples_with_metadata", preview_row)

if "Codes" in prompt_examples_df.columns and prompt_examples_df["Codes"].fillna("").astype(str).str.strip().ne("").any():
    assert "Codes:" in no_meta_prompt, "Human-coded Codes labels are missing from no-metadata examples."

for metadata_label in ["Rationale:", "Reference:", "Source:"]:
    assert metadata_label not in no_meta_prompt, f"{metadata_label} leaked into the no-metadata prompt."

available_metadata_labels = [
    f"{col}:"
    for col in ["Reference", "Rationale", "Source"]
    if col in prompt_examples_df.columns
    and prompt_examples_df[col].fillna("").astype(str).str.strip().ne("").any()
]
for metadata_label in available_metadata_labels:
    assert metadata_label in with_meta_prompt, f"{metadata_label} is missing from the metadata prompt."

assert "confidence" in AnnotationOutput.model_fields
assert "confidence" in ANNOTATION_JSON_SCHEMA.get("properties", {})

prompt_audit_rows = []
for variant in PROMPT_VARIANTS_TO_RUN:
    messages = build_prompt_messages(variant, preview_row)
    user_prompt = messages[1]["content"]
    prompt_audit_rows.append({
        "prompt_version": PROMPT_VERSION,
        "prompt_variant": variant,
        "system_prompt": messages[0]["content"],
        "prompt_sha256_first_row": prompt_sha256(messages),
        "user_prompt_chars_first_row": len(user_prompt),
        "contains_codebook": "Codebook:" in user_prompt,
        "contains_codebook_examples_field": "Examples:" in user_prompt,
        "contains_labeled_examples": "Labeled examples:" in user_prompt,
        "contains_human_codes": "Codes:" in user_prompt,
        "contains_example_rationale": "Rationale:" in user_prompt,
        "requires_confidence": '"confidence"' in user_prompt,
    })

prompt_audit_df = pd.DataFrame(prompt_audit_rows)
print("Prompt/schema assertions: PASSED")
display(prompt_audit_df)

for variant in PROMPT_VARIANTS_TO_RUN:
    messages = build_prompt_messages(variant, preview_row)
    print("=" * 100)
    print("PROMPT VARIANT:", variant)
    print("PROMPT SHA256:", prompt_sha256(messages))
    print(messages[1]["content"][:3500])
    print("...\n")


Prompt/schema assertions: PASSED


,prompt_version,prompt_variant,system_prompt,prompt_sha256_first_row,user_prompt_chars_first_row,contains_codebook,contains_codebook_examples_field,contains_labeled_examples,contains_human_codes,contains_example_rationale,requires_confidence
0,v4_provider_separated_confidence_api_fixes,direct_no_context,Return valid JSON only. Do not include markdown.,dfa06a441ee3c990d61ef433434962ffd422a900dc680a...,1073,False,False,False,False,False,True
1,v4_provider_separated_confidence_api_fixes,definitions_only,Return valid JSON only. Do not include markdown.,7be60d0f27c8efc942ef54277ff9917b612413ed3d50bc...,11765,True,True,False,False,False,True
2,v4_provider_separated_confidence_api_fixes,definitions_examples_no_metadata,Return valid JSON only. Do not include markdown.,e7a4fa673c91c5a461001bf49dc85e1154723503139043...,18242,True,True,True,True,False,True
3,v4_provider_separated_confidence_api_fixes,definitions_examples_with_metadata,Return valid JSON only. Do not include markdown.,f16ff7ee7b51fb0ce78842fa9edee048626c3ffec37088...,18943,True,True,True,True,True,True


PROMPT VARIANT: direct_no_context
PROMPT SHA256: dfa06a441ee3c990d61ef433434962ffd422a900dc680ade6069437ac57895e8
You are coding federal disaster policy text using the unlearning codebook.

Task:
1. Decide whether the paragraph contains unlearning.
2. If yes, assign one target type.
3. Identify the agency or sub-unit if the paragraph states one.
4. Keep the rationale short.

Return valid JSON only with this structure:
{
  "unlearning_present": true/false,
  "target_type": "Leadership | laws_plans_policies | capabilities | funds_resources | misc_organizational | none",
  "agency": "agency or sub-unit name if stated, otherwise null",
  "confidence": "number from 0.0 to 1.0",
  "rationale": "one short sentence"
}

Paragraph ID: 14:2
Paragraph:
Because of FEMA’s mission performance during Hurricane Katrina, concerns have been raised regarding the agency’s organizational placement, including whether it should be disbanded and functions moved to other agencies, remain within the Department o

## 6. Token and cost estimation

Actual token usage comes from each API response after you run the model. Before spending credits, this section estimates cost using prompt length and model pricing.

In [ ]:
def estimate_tokens_text(text: str, provider: str = "generic") -> int:
    """Best-effort token estimate for preflight budgeting."""
    if text is None:
        return 0
    text = str(text)
    if provider == "openai":
        try:
            import tiktoken
            enc = tiktoken.get_encoding("cl100k_base")
            return len(enc.encode(text))
        except Exception:
            pass
    return max(1, math.ceil(len(text) / 4))


def estimate_messages_tokens(messages: list[dict], provider: str = "generic") -> int:
    return sum(estimate_tokens_text(m.get("content", ""), provider) + 6 for m in messages) + 10


def cost_from_tokens(input_tokens: int, output_tokens: int, cfg: dict) -> float:
    return (
        (input_tokens / 1_000_000) * cfg["input_usd_per_1m"]
        + (output_tokens / 1_000_000) * cfg["output_usd_per_1m"]
    )


def model_effort_descriptor(cfg: dict) -> str:
    if cfg["provider"] == "openai":
        return f"reasoning_effort={cfg.get('reasoning_effort', 'none')}"
    if cfg["provider"] == "anthropic":
#        return f"effort={cfg.get('effort', 'high')}"
        if cfg.get("effort") is not None:
                return f"effort={cfg['effort']}"

        return "effort=omitted"

    if cfg["provider"] == "gemini":
        return f"thinking_level={cfg.get('thinking_level', 'low')}"
    return "default"


def active_model_configs(require_api_key: bool = False) -> list[dict]:
    active = []
    for cfg in MODEL_CONFIG:
        if not cfg.get("enabled", True):
            continue
        key_present = bool(os.getenv(cfg.get("api_key_env", "")))
        if require_api_key and not key_present:
            print(f"Skipping {cfg['provider']} / {cfg['model']} because {cfg['api_key_env']} is not set.")
            continue
        active.append(cfg)
    return active


def estimate_full_grid(
    test_rows: pd.DataFrame,
    model_configs: list[dict],
    variants: list[str],
    assumed_output_tokens: int = 100,
) -> pd.DataFrame:
    rows = []
    for cfg in model_configs:
        for variant in variants:
            input_total = 0
            for _, row in test_rows.iterrows():
                messages = build_prompt_messages(variant, row)
                input_total += estimate_messages_tokens(messages, provider=cfg["provider"])
            output_total = assumed_output_tokens * len(test_rows)
            rows.append({
                "provider": cfg["provider"],
                "model": cfg["model"],
                "model_effort": model_effort_descriptor(cfg),
                "prompt_version": PROMPT_VERSION,
                "prompt_variant": variant,
                "n_rows": len(test_rows),
                "estimated_input_tokens": int(input_total),
                "assumed_output_tokens": int(output_total),
                "estimated_total_tokens": int(input_total + output_total),
                "estimated_cost_usd": cost_from_tokens(input_total, output_total, cfg),
            })
    return pd.DataFrame(rows)


estimate_df = estimate_full_grid(
    test_df,
    active_model_configs(require_api_key=False),
    PROMPT_VARIANTS_TO_RUN,
)
display(estimate_df)

provider_estimates = estimate_df.groupby(
    ["provider", "model", "model_effort"],
    as_index=False,
).agg(
    estimated_input_tokens=("estimated_input_tokens", "sum"),
    assumed_output_tokens=("assumed_output_tokens", "sum"),
    estimated_total_tokens=("estimated_total_tokens", "sum"),
    estimated_cost_usd=("estimated_cost_usd", "sum"),
)
display(provider_estimates)
print("Estimated total cost across all enabled models and prompts: $", round(estimate_df["estimated_cost_usd"].sum(), 6))


,provider,model,model_effort,prompt_version,prompt_variant,n_rows,estimated_input_tokens,assumed_output_tokens,estimated_total_tokens,estimated_cost_usd
0,openai,gpt-5.4,reasoning_effort=low,v4_provider_separated_confidence_api_fixes,direct_no_context,11,4133,1100,5233,0.002202
1,openai,gpt-5.4,reasoning_effort=low,v4_provider_separated_confidence_api_fixes,definitions_only,11,25847,1100,26947,0.006544
2,openai,gpt-5.4,reasoning_effort=low,v4_provider_separated_confidence_api_fixes,definitions_examples_no_metadata,11,39355,1100,40455,0.009246
3,openai,gpt-5.4,reasoning_effort=low,v4_provider_separated_confidence_api_fixes,definitions_examples_with_metadata,11,41654,1100,42754,0.009706
4,anthropic,claude-haiku-4-5-20251001,effort=omitted,v4_provider_separated_confidence_api_fixes,direct_no_context,11,5164,1100,6264,0.010664
5,anthropic,claude-haiku-4-5-20251001,effort=omitted,v4_provider_separated_confidence_api_fixes,definitions_only,11,34567,1100,35667,0.040067
6,anthropic,claude-haiku-4-5-20251001,effort=omitted,v4_provider_separated_confidence_api_fixes,definitions_examples_no_metadata,11,52377,1100,53477,0.057877
7,anthropic,claude-haiku-4-5-20251001,effort=omitted,v4_provider_separated_confidence_api_fixes,definitions_examples_with_metadata,11,54305,1100,55405,0.059805
8,gemini,gemini-3.5-flash,thinking_level=low,v4_provider_separated_confidence_api_fixes,direct_no_context,11,5164,1100,6264,0.017646
9,gemini,gemini-3.5-flash,thinking_level=low,v4_provider_separated_confidence_api_fixes,definitions_only,11,34567,1100,35667,0.061750


,provider,model,model_effort,estimated_input_tokens,assumed_output_tokens,estimated_total_tokens,estimated_cost_usd
0,anthropic,claude-haiku-4-5-20251001,effort=omitted,146413,4400,150813,0.168413
1,gemini,gemini-3.5-flash,thinking_level=low,146413,4400,150813,0.259220
2,openai,gpt-5.4,reasoning_effort=low,110989,4400,115389,0.027698


Estimated total cost across all enabled models and prompts: $ 0.45533


## 7. API wrappers

All provider wrappers return the same normalized contract:

```python
{
    "raw_response": str,
    "input_tokens": int,
    "output_tokens": int,
    "total_tokens": int,
    "latency_seconds": float,
    "reasoning_tokens": int,
}
```

This fixes the prior Claude wrapper mismatch, where Claude returned `raw_text` while the shared grid expected `raw_response`.

The OpenAI call uses the Responses API with Structured Outputs and explicit low reasoning effort. Claude uses Structured Outputs plus medium effort. Gemini uses JSON-schema-constrained output and its default dynamic thinking behavior.


In [ ]:
def extract_json_object(text: str) -> dict:
    """Parse a JSON object from a response, with a conservative fallback."""
    if isinstance(text, dict):
        return text
    s = str(text or "").strip()
    if not s:
        return {}

    try:
        parsed = json.loads(s)
        return parsed if isinstance(parsed, dict) else {}
    except json.JSONDecodeError:
        pass

    match = re.search(r"\{.*\}", s, flags=re.DOTALL)
    if not match:
        return {}

    try:
        parsed = json.loads(match.group(0))
        return parsed if isinstance(parsed, dict) else {}
    except json.JSONDecodeError:
        return {}


def normalize_model_prediction(parsed: dict, raw_text: str = ""):
    """Normalize the common structured output or conservative legacy fallbacks to Yes/No."""
    if isinstance(parsed, dict):
        if "unlearning_present" in parsed:
            return normalize_yes_no(parsed.get("unlearning_present"))

        for key in ["unlearning", "prediction", "label", "answer", "classification"]:
            if key in parsed:
                normalized = normalize_yes_no(parsed.get(key))
                if not pd.isna(normalized):
                    return normalized

    s = str(raw_text or "").strip()
    field_match = re.search(
        r'"?(?:unlearning_present|unlearning|label|prediction|answer|classification)"?\s*:\s*"?(yes|no|true|false)"?',
        s,
        flags=re.IGNORECASE,
    )
    if field_match:
        return normalize_yes_no(field_match.group(1))

    if len(s) < 200:
        yn_match = re.search(r"\b(yes|no)\b", s, flags=re.IGNORECASE)
        if yn_match:
            return normalize_yes_no(yn_match.group(1))

    return np.nan


def normalize_confidence(value):
    """Return a float in [0, 1], otherwise NaN."""
    try:
        number = float(value)
    except (TypeError, ValueError):
        return np.nan
    return number if 0.0 <= number <= 1.0 else np.nan


class APIErrorForRetry(Exception):
    pass


class NonRetryableQuotaError(RuntimeError):
    pass


class NonRetryableRequestError(RuntimeError):
    pass


def _retry_wait_from_error(error: Exception, default_wait: float) -> float:
    """Honor provider retry-delay hints such as retryDelay:'36s' or 'Please retry in 36.87s'."""
    text = str(error)
    patterns = [
        r"retryDelay[^0-9]*(\d+(?:\.\d+)?)s",
        r"Please retry in\s+(\d+(?:\.\d+)?)s",
        r"retry in\s+(\d+(?:\.\d+)?)\s*seconds?",
    ]
    for pattern in patterns:
        match = re.search(pattern, text, flags=re.IGNORECASE)
        if match:
            return float(match.group(1)) + 1.0
    return default_wait


def _is_nonretryable_request_error(error: Exception) -> bool:
    """Do not waste retries on malformed payloads, deprecated parameters, or schema errors."""
    text = str(error).lower()
    request_markers = [
        "error code: 400",
        "400 invalid_argument",
        "invalid_request_error",
        "invalid json payload",
        "unknown name",
        "deprecated for this model",
        "cannot find field",
    ]
    return any(marker in text for marker in request_markers)


def retry_api_call(max_attempts=4, min_wait=5, max_wait=90):
    def decorator(fn):
        def wrapper(*args, **kwargs):
            last_error = None

            for attempt in range(max_attempts):
                try:
                    return fn(*args, **kwargs)

                except (
                    NonRetryableQuotaError,
                    NonRetryableRequestError,
                ):
                    raise

                except Exception as e:
                    last_error = e

                    # Malformed requests/configuration errors
                    # cannot be fixed by waiting.
                    if _is_nonretryable_request_error(e):
                        raise NonRetryableRequestError(
                            "Non-retryable API "
                            "request/configuration error in "
                            f"{fn.__name__}: {e}"
                        ) from e

                    # On paid Gemini, quota/rate-limit 429s are
                    # treated as transient and retried using the
                    # provider's retry-delay hint.
                    if attempt == max_attempts - 1:
                        raise

                    default_wait = min(
                        max_wait,
                        min_wait * (2 ** attempt),
                    )

                    wait_seconds = min(
                        max_wait,
                        _retry_wait_from_error(
                            e,
                            default_wait,
                        ),
                    )

                    print(
                        "Retrying after transient error from "
                        f"{fn.__name__}: {e}\n"
                        f"Waiting {wait_seconds:.1f}s..."
                    )

                    time.sleep(wait_seconds)

            raise last_error

        return wrapper

    return decorator


def _common_api_result(
    *,
    raw_response: str,
    input_tokens: int,
    output_tokens: int,
    total_tokens: int,
    latency_seconds: float,
    reasoning_tokens: int = 0,
) -> dict:
    return {
        "raw_response": str(raw_response or ""),
        "input_tokens": int(input_tokens or 0),
        "output_tokens": int(output_tokens or 0),
        "total_tokens": int(total_tokens or (input_tokens or 0) + (output_tokens or 0)),
        "latency_seconds": float(latency_seconds),
        "reasoning_tokens": int(reasoning_tokens or 0),
    }


@retry_api_call(max_attempts=3, min_wait=5, max_wait=45)
def call_openai(messages: list[dict], cfg: dict) -> dict:
    """GPT-5.4 nano via Responses API + Pydantic Structured Outputs + explicit reasoning effort."""
    from openai import OpenAI

    client = OpenAI(api_key=os.getenv(cfg["api_key_env"]))
    system_text = next((m["content"] for m in messages if m["role"] == "system"), "")
    response_input = [m for m in messages if m["role"] != "system"]

    start = time.time()
    response = client.responses.parse(
        model=cfg["model"],
        instructions=system_text,
        input=response_input,
        reasoning={"effort": cfg.get("reasoning_effort", "low")},
        max_output_tokens=MAX_OUTPUT_TOKENS,
        text_format=AnnotationOutput,
        store=False,
    )
    latency = time.time() - start

    parsed_obj = response.output_parsed
    if parsed_obj is None:
        raise APIErrorForRetry(
            f"OpenAI returned no parsed structured output. Raw output: {response.output_text!r}"
        )

    raw_text = json.dumps(parsed_obj.model_dump(), ensure_ascii=False)

    usage = getattr(response, "usage", None)
    input_tokens = getattr(usage, "input_tokens", 0) or 0
    output_tokens = getattr(usage, "output_tokens", 0) or 0
    total_tokens = getattr(usage, "total_tokens", 0) or input_tokens + output_tokens

    output_details = getattr(usage, "output_tokens_details", None)
    reasoning_tokens = getattr(output_details, "reasoning_tokens", 0) or 0

    return _common_api_result(
        raw_response=raw_text,
        input_tokens=input_tokens,
        output_tokens=output_tokens,
        total_tokens=total_tokens,
        latency_seconds=latency,
        reasoning_tokens=reasoning_tokens,
    )


@retry_api_call(max_attempts=3, min_wait=5, max_wait=45)
def call_anthropic(messages: list[dict], cfg: dict) -> dict:
    """Claude Sonnet 5 via messages.parse(), Pydantic Structured Outputs, and explicit effort."""
    from anthropic import Anthropic

    client = Anthropic(api_key=os.environ[cfg["api_key_env"]])

    system_text = "\n".join(
        m.get("content", "")
        for m in messages
        if m.get("role") == "system"
    ).strip()

    anthropic_messages = [
        {"role": m["role"], "content": m.get("content", "")}
        for m in messages
        if m.get("role") in {"user", "assistant"}
    ]

#    start = time.time()
#    # IMPORTANT: do not pass temperature to Claude Sonnet 5. The live API rejects it as deprecated.
#    response = client.messages.parse(
#        model=cfg["model"],
#        max_tokens=MAX_OUTPUT_TOKENS,
#        system=system_text,
#        messages=anthropic_messages,
#        output_config={"effort": cfg.get("effort", "medium")},
#        output_format=AnnotationOutput,
#    )
#    latency = time.time() - start


    start = time.time()

    anthropic_request = {
        "model": cfg["model"],
        "max_tokens": MAX_OUTPUT_TOKENS,
        "system": system_text,
        "messages": anthropic_messages,
        "output_format": AnnotationOutput,
    }

    # Sonnet 5 supports effort.
    # Haiku 4.5 does not.
    if cfg.get("effort") is not None:
        anthropic_request["output_config"] = {
            "effort": cfg["effort"]
        }

    response = client.messages.parse(
        **anthropic_request
    )

    latency = time.time() - start


    parsed_obj = response.parsed_output
    if parsed_obj is None:
        raise APIErrorForRetry("Anthropic returned no parsed structured output.")

    raw_text = json.dumps(parsed_obj.model_dump(), ensure_ascii=False)

    usage = getattr(response, "usage", None)
    input_tokens = getattr(usage, "input_tokens", 0) or 0
    output_tokens = getattr(usage, "output_tokens", 0) or 0

    return _common_api_result(
        raw_response=raw_text,
        input_tokens=input_tokens,
        output_tokens=output_tokens,
        total_tokens=input_tokens + output_tokens,
        latency_seconds=latency,
        reasoning_tokens=0,
    )


@retry_api_call(max_attempts=4, min_wait=8, max_wait=120)
def call_gemini(messages: list[dict], cfg: dict) -> dict:
    """Gemini 2.5 Flash via generateContent + native raw JSON Schema + explicit thinking budget."""
    from google import genai
    from google.genai import types

    client = genai.Client(api_key=os.getenv(cfg["api_key_env"]))
    system_text = next((m["content"] for m in messages if m["role"] == "system"), "")
    user_text = "\n\n".join(
        m["content"]
        for m in messages
        if m["role"] == "user"
    )

    start = time.time()
    response = client.models.generate_content(
        model=cfg["model"],
        contents=user_text,
        config=types.GenerateContentConfig(
            system_instruction=system_text,
            max_output_tokens=MAX_OUTPUT_TOKENS,
            response_mime_type="application/json",
            # response_json_schema sends raw JSON Schema. Do NOT use response_schema with model_json_schema().
            response_json_schema=ANNOTATION_JSON_SCHEMA,
            thinking_config=types.ThinkingConfig(
                thinking_level=cfg.get("thinking_level", "low")
            ),
        ),
    )
    latency = time.time() - start


    raw_text = response.text or ""

    # Diagnose why Gemini stopped generating.
    finish_reason = None
    if getattr(response, "candidates", None):
        finish_reason = getattr(response.candidates[0], "finish_reason", None)

    finish_reason_text = str(finish_reason or "")

    if "MAX_TOKENS" in finish_reason_text.upper():
        raise NonRetryableRequestError(
            "Gemini output was truncated because max_output_tokens was reached. "
            f"finish_reason={finish_reason_text}. "
            f"Raw partial response: {raw_text!r}"
        )

    try:
        parsed_obj = AnnotationOutput.model_validate_json(raw_text)
    except ValidationError as e:
        raise APIErrorForRetry(
            "Gemini returned complete-looking output that failed "
            f"AnnotationOutput validation: {e}. "
            f"finish_reason={finish_reason_text}. "
            f"Raw: {raw_text!r}"
        ) from e


    raw_text = json.dumps(parsed_obj.model_dump(), ensure_ascii=False)

    usage = getattr(response, "usage_metadata", None)
    input_tokens = getattr(usage, "prompt_token_count", 0) or 0
    output_tokens = getattr(usage, "candidates_token_count", 0) or 0
    total_tokens = getattr(usage, "total_token_count", 0) or input_tokens + output_tokens
    thinking_tokens = getattr(usage, "thoughts_token_count", 0) or 0

    return _common_api_result(
        raw_response=raw_text,
        input_tokens=input_tokens,
        output_tokens=output_tokens,
        total_tokens=total_tokens,
        latency_seconds=latency,
        reasoning_tokens=thinking_tokens,
    )


def call_model(messages: list[dict], cfg: dict) -> dict:
    provider = cfg["provider"].lower()
    if provider == "openai":
        return call_openai(messages, cfg)
    if provider == "anthropic":
        return call_anthropic(messages, cfg)
    if provider == "gemini":
        return call_gemini(messages, cfg)
    raise ValueError(f"Unsupported provider: {provider}")


## 8. Provider-separated execution

Each provider below has:

- its own execution cell;
- its own JSONL audit/resume log;
- its own prediction CSV snapshot saved immediately after that provider finishes or stops;
- its own status/error diagnostics.

Run **8A OpenAI**, inspect the output, then **8B Claude**, then **8C Gemini**. A 400/configuration error is non-retryable and the provider stops after the first final error when `STOP_PROVIDER_ON_FIRST_ERROR=True`.


In [ ]:
def get_provider_config(provider: str) -> dict:
    matches = [cfg for cfg in MODEL_CONFIG if cfg["provider"].lower() == provider.lower()]
    if len(matches) != 1:
        raise ValueError(f"Expected exactly one config for {provider!r}; found {len(matches)}.")
    return matches[0]


def provider_results_jsonl(provider: str) -> Path:
    return OUTPUT_DIR / f"{provider}_raw_results_{PROMPT_VERSION}.jsonl"


def provider_predictions_csv(provider: str) -> Path:
    return OUTPUT_DIR / f"{provider}_predictions_{PROMPT_VERSION}.csv"


def make_run_key(cfg: dict, prompt_variant: str, row_id: str, prompt_hash: str) -> str:
    signature = {
        "provider": cfg["provider"],
        "model": cfg["model"],
        "model_effort": model_effort_descriptor(cfg),
        "prompt_variant": prompt_variant,
        "row_id": str(row_id),
        "prompt_version": PROMPT_VERSION,
        "prompt_sha256": prompt_hash,
    }
    canonical = json.dumps(signature, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(canonical.encode("utf-8")).hexdigest()


def load_result_state(path: Path):
    """Return latest record per run key, successful resume keys, and prior attempt counts."""
    latest_by_key = {}
    success_keys = set()
    attempt_counts = defaultdict(int)

    if not path.exists():
        return latest_by_key, success_keys, attempt_counts

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            rec = json.loads(line)
            run_key = rec.get("run_key")
            if not run_key:
                continue
            attempt_counts[run_key] += 1
            latest_by_key[run_key] = rec
            if rec.get("status") == "ok":
                success_keys.add(run_key)

    return latest_by_key, success_keys, attempt_counts


def append_jsonl(path: Path, record: dict):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def build_record_from_api_result(base_record: dict, api_result: dict, cfg: dict, estimated_input_tokens: int) -> dict:
    parsed = extract_json_object(api_result["raw_response"])
    pred = normalize_model_prediction(parsed, api_result["raw_response"])
    confidence = normalize_confidence(parsed.get("confidence")) if isinstance(parsed, dict) else np.nan

    pred_target_type = parsed.get("target_type", "") if isinstance(parsed, dict) else ""
    pred_agency = parsed.get("agency", "") if isinstance(parsed, dict) else ""
    rationale = parsed.get("rationale", "") if isinstance(parsed, dict) else ""

    input_tokens = api_result["input_tokens"] or estimated_input_tokens
    output_tokens = api_result["output_tokens"]
    total_tokens = api_result["total_tokens"] or input_tokens + output_tokens
    cost_usd = cost_from_tokens(input_tokens, output_tokens, cfg)

    parse_problems = []
    if pd.isna(pred):
        parse_problems.append("Could not parse unlearning_present")
    if pd.isna(confidence):
        parse_problems.append("Missing/invalid confidence")

    return {
        **base_record,
        "status": "ok" if not parse_problems else "parse_failed",
        "raw_response": api_result["raw_response"],
        "pred_unlearning": pred,
        "pred_target_type": pred_target_type,
        "pred_agency": pred_agency,
        "confidence": confidence,
        "rationale": rationale,
        "input_tokens": int(input_tokens),
        "output_tokens": int(output_tokens),
        "reasoning_tokens": int(api_result.get("reasoning_tokens", 0)),
        "total_tokens": int(total_tokens),
        "cost_usd": float(cost_usd),
        "latency_seconds": float(api_result["latency_seconds"]),
        "parse_error": "; ".join(parse_problems),
    }


def run_single_provider(
    test_rows: pd.DataFrame,
    cfg: dict,
    variants: list[str],
    results_jsonl: Path,
    run_api_calls: bool,
) -> pd.DataFrame:
    latest_by_key, success_keys, attempt_counts = (
        load_result_state(results_jsonl)
        if RESUME_FROM_JSONL
        else ({}, set(), defaultdict(int))
    )

    print(f"Provider: {cfg['provider']} / {cfg['model']}")
    print(f"Resume log: {results_jsonl}")
    print(f"Prior unique run keys: {len(latest_by_key)}; successful resume keys: {len(success_keys)}")

    total_calls = len(test_rows) * len(variants)
    if not run_api_calls:
        print("Provider run switch is False: generating dry-run records only; no API calls will be made.")

    with tqdm(total=total_calls, desc=f"{cfg['provider']} A/B") as pbar:
        for variant in variants:
            for _, row in test_rows.iterrows():
                row_id = str(row.get("Number", row.name))
                messages = build_prompt_messages(variant, row)
                prompt_hash = prompt_sha256(messages)
                run_key = make_run_key(cfg, variant, row_id, prompt_hash)

                if run_key in success_keys:
                    pbar.update(1)
                    continue

                estimated_input_tokens = estimate_messages_tokens(messages, provider=cfg["provider"])
                attempt_number = int(attempt_counts[run_key]) + 1

                base_record = {
                    "run_key": run_key,
                    "attempt_number": attempt_number,
                    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
                    "provider": cfg["provider"],
                    "model": cfg["model"],
                    "model_effort": model_effort_descriptor(cfg),
                    "prompt_version": PROMPT_VERSION,
                    "prompt_sha256": prompt_hash,
                    "prompt_variant": variant,
                    "row_id": row_id,
                    "document": row.get("Document", ""),
                    "text_content": row.get("Text Content", ""),
                    "ground_truth_unlearning": row.get("ground_truth_unlearning", np.nan),
                    "estimated_input_tokens": int(estimated_input_tokens),
                }

                if not run_api_calls:
                    rec = {
                        **base_record,
                        "status": "dry_run",
                        "raw_response": "",
                        "pred_unlearning": np.nan,
                        "pred_target_type": "",
                        "pred_agency": "",
                        "confidence": np.nan,
                        "rationale": "",
                        "input_tokens": 0,
                        "output_tokens": 0,
                        "reasoning_tokens": 0,
                        "total_tokens": 0,
                        "cost_usd": 0.0,
                        "latency_seconds": 0.0,
                        "parse_error": "",
                    }
                else:
                    try:
                        api_result = call_model(messages, cfg)
                        rec = build_record_from_api_result(base_record, api_result, cfg, estimated_input_tokens)
                    except Exception as e:
                        rec = {
                            **base_record,
                            "status": "error",
                            "raw_response": "",
                            "pred_unlearning": np.nan,
                            "pred_target_type": "",
                            "pred_agency": "",
                            "confidence": np.nan,
                            "rationale": "",
                            # We have no provider usage object on a failed request; do not mislabel estimates as actual usage/cost.
                            "input_tokens": 0,
                            "output_tokens": 0,
                            "reasoning_tokens": 0,
                            "total_tokens": 0,
                            "cost_usd": np.nan,
                            "latency_seconds": 0.0,
                            "parse_error": f"{type(e).__name__}: {e}",
                        }

                append_jsonl(results_jsonl, rec)
                attempt_counts[run_key] += 1
                latest_by_key[run_key] = rec

                if rec["status"] == "ok":
                    success_keys.add(run_key)

                pbar.update(1)

                if rec["status"] == "error" and STOP_PROVIDER_ON_FIRST_ERROR:
                    print("\nProvider stopped after first final API error.")
                    print("Variant:", variant, "Row:", row_id)
                    print("Error:", rec["parse_error"])
                    result_df = pd.DataFrame(latest_by_key.values())
                    result_df.to_csv(provider_predictions_csv(cfg["provider"]), index=False)
                    return result_df

                time.sleep(cfg.get("request_sleep_seconds", REQUEST_SLEEP_SECONDS))

    result_df = pd.DataFrame(latest_by_key.values())
    result_df.to_csv(provider_predictions_csv(cfg["provider"]), index=False)
    return result_df


def provider_diagnostics(df: pd.DataFrame, provider: str):
    print(f"\n{provider.upper()} status counts:")
    if df.empty:
        print("No records.")
        return
    display(df["status"].value_counts(dropna=False).rename_axis("status").reset_index(name="n"))

    failures = df[df["status"].isin(["error", "parse_failed"])].copy()
    if not failures.empty:
        print(f"{provider.upper()} failures / parse failures:")
        display(failures[[
            "prompt_variant", "row_id", "attempt_number", "status", "parse_error", "raw_response"
        ]].head(20))

    ok = df[df["status"].eq("ok")].copy()
    if not ok.empty:
        print(f"{provider.upper()} successful predictions preview:")
        display(ok[[
            "prompt_variant", "row_id", "pred_unlearning", "pred_target_type",
            "pred_agency", "confidence", "rationale", "input_tokens", "output_tokens",
            "reasoning_tokens", "cost_usd"
        ]].head(20))


### 8A. Run OpenAI / GPT-5.4 nano only

This cell calls only OpenAI and writes only the OpenAI JSONL/CSV files. Re-running it resumes successful OpenAI rows and retries only unfinished/failed OpenAI run keys.


In [ ]:
#5.4 nano
OPENAI_CFG = get_provider_config("openai")
openai_preflight_df = estimate_full_grid(test_df, [OPENAI_CFG], PROMPT_VARIANTS_TO_RUN)
openai_estimated_cost = openai_preflight_df["estimated_cost_usd"].sum()
print("OpenAI preflight estimated cost: $", round(openai_estimated_cost, 6))
display(openai_preflight_df)

if RUN_OPENAI_CALLS and not os.getenv(OPENAI_CFG["api_key_env"]):
    raise RuntimeError(f"{OPENAI_CFG['api_key_env']} is not set.")
if RUN_OPENAI_CALLS and openai_estimated_cost > MAX_ESTIMATED_COST_USD_PER_PROVIDER:
    raise RuntimeError(
        f"OpenAI estimated cost ${openai_estimated_cost:.4f} exceeds per-provider guard "
        f"${MAX_ESTIMATED_COST_USD_PER_PROVIDER:.2f}."
    )

OPENAI_RESULTS_JSONL = provider_results_jsonl("openai")
results_openai_df = run_single_provider(
    test_df,
    OPENAI_CFG,
    PROMPT_VARIANTS_TO_RUN,
    OPENAI_RESULTS_JSONL,
    RUN_OPENAI_CALLS,
)
print("OpenAI snapshot:", provider_predictions_csv("openai"))
provider_diagnostics(results_openai_df, "openai")


OpenAI preflight estimated cost: $ 0.027698


,provider,model,model_effort,prompt_version,prompt_variant,n_rows,estimated_input_tokens,assumed_output_tokens,estimated_total_tokens,estimated_cost_usd
0,openai,gpt-5.4-nano,reasoning_effort=medium,v4_provider_separated_confidence_api_fixes,direct_no_context,11,4133,1100,5233,0.002202
1,openai,gpt-5.4-nano,reasoning_effort=medium,v4_provider_separated_confidence_api_fixes,definitions_only,11,25847,1100,26947,0.006544
2,openai,gpt-5.4-nano,reasoning_effort=medium,v4_provider_separated_confidence_api_fixes,definitions_examples_no_metadata,11,39355,1100,40455,0.009246
3,openai,gpt-5.4-nano,reasoning_effort=medium,v4_provider_separated_confidence_api_fixes,definitions_examples_with_metadata,11,41654,1100,42754,0.009706


Provider: openai / gpt-5.4-nano
Resume log: outputs_unlearning_ab_test_v4/openai_raw_results_v4_provider_separated_confidence_api_fixes.jsonl
Prior unique run keys: 0; successful resume keys: 0


openai A/B:   0%|          | 0/44 [00:00<?, ?it/s]

Retrying after transient error from call_openai: 1 validation error for AnnotationOutput
  Invalid JSON: EOF while parsing a string at line 1 column 254 [type=json_invalid, input_value='{"unlearning_present":tr...ity to activate Reserve', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/json_invalid
Waiting 5.0s...
Retrying after transient error from call_openai: 1 validation error for AnnotationOutput
  Invalid JSON: EOF while parsing a string at line 1 column 273 [type=json_invalid, input_value='{"unlearning_present":fa...y Katrina-era policy or', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/json_invalid
Waiting 5.0s...
OpenAI snapshot: outputs_unlearning_ab_test_v4/openai_predictions_v4_provider_separated_confidence_api_fixes.csv

OPENAI status counts:


,status,n
0,ok,44


OPENAI successful predictions preview:


,prompt_variant,row_id,pred_unlearning,pred_target_type,pred_agency,confidence,rationale,input_tokens,output_tokens,reasoning_tokens,cost_usd
0,direct_no_context,14:2,Yes,misc_organizational,FEMA,0.66,The paragraph raises whether FEMA should be di...,385,192,127,0.000317
1,direct_no_context,14:3,No,none,FEMA,0.82,The paragraph describes GAO reviews and policy...,640,267,205,0.000462
2,direct_no_context,14:4,Yes,misc_organizational,FEMA,0.62,The paragraph raises concerns about FEMA’s org...,400,171,97,0.000294
3,direct_no_context,14:5,Yes,capabilities,The White House,0.62,The text argues against relying on the traditi...,443,228,166,0.000374
4,direct_no_context,14:6,No,none,NaN,0.84,The paragraph discusses improving disaster res...,436,87,31,0.000196
5,direct_no_context,14:7,Yes,laws_plans_policies,NaN,0.78,It calls for replacing unquestioned adherence ...,444,114,49,0.000231
6,direct_no_context,14:8,No,none,null,0.78,The paragraph discusses planning and recommend...,523,165,103,0.000311
7,direct_no_context,14:9,No,none,Department of Defense (DOD),0.62,The paragraph describes assessment capability ...,425,202,132,0.000338
8,direct_no_context,14:12,No,none,NaN,0.74,The paragraph argues that organizational place...,853,176,114,0.000391
9,direct_no_context,14:13,No,none,NaN,0.86,The paragraph describes damage and rebuilding ...,416,83,26,0.000187


In [ ]:
#5.4 mini
OPENAI_CFG = get_provider_config("openai")
openai_preflight_df = estimate_full_grid(test_df, [OPENAI_CFG], PROMPT_VARIANTS_TO_RUN)
openai_estimated_cost = openai_preflight_df["estimated_cost_usd"].sum()
print("OpenAI preflight estimated cost: $", round(openai_estimated_cost, 6))
display(openai_preflight_df)

if RUN_OPENAI_CALLS and not os.getenv(OPENAI_CFG["api_key_env"]):
    raise RuntimeError(f"{OPENAI_CFG['api_key_env']} is not set.")
if RUN_OPENAI_CALLS and openai_estimated_cost > MAX_ESTIMATED_COST_USD_PER_PROVIDER:
    raise RuntimeError(
        f"OpenAI estimated cost ${openai_estimated_cost:.4f} exceeds per-provider guard "
        f"${MAX_ESTIMATED_COST_USD_PER_PROVIDER:.2f}."
    )

OPENAI_RESULTS_JSONL = provider_results_jsonl("openai")
results_openai_df = run_single_provider(
    test_df,
    OPENAI_CFG,
    PROMPT_VARIANTS_TO_RUN,
    OPENAI_RESULTS_JSONL,
    RUN_OPENAI_CALLS,
)
print("OpenAI snapshot:", provider_predictions_csv("openai"))
provider_diagnostics(results_openai_df, "openai")


OpenAI preflight estimated cost: $ 0.027698


,provider,model,model_effort,prompt_version,prompt_variant,n_rows,estimated_input_tokens,assumed_output_tokens,estimated_total_tokens,estimated_cost_usd
0,openai,gpt-5.4-mini,reasoning_effort=low,v4_provider_separated_confidence_api_fixes,direct_no_context,11,4133,1100,5233,0.002202
1,openai,gpt-5.4-mini,reasoning_effort=low,v4_provider_separated_confidence_api_fixes,definitions_only,11,25847,1100,26947,0.006544
2,openai,gpt-5.4-mini,reasoning_effort=low,v4_provider_separated_confidence_api_fixes,definitions_examples_no_metadata,11,39355,1100,40455,0.009246
3,openai,gpt-5.4-mini,reasoning_effort=low,v4_provider_separated_confidence_api_fixes,definitions_examples_with_metadata,11,41654,1100,42754,0.009706


Provider: openai / gpt-5.4-mini
Resume log: outputs_unlearning_ab_test_v4/openai_raw_results_v4_provider_separated_confidence_api_fixes.jsonl
Prior unique run keys: 44; successful resume keys: 44


openai A/B:   0%|          | 0/44 [00:00<?, ?it/s]

OpenAI snapshot: outputs_unlearning_ab_test_v4/openai_predictions_v4_provider_separated_confidence_api_fixes.csv

OPENAI status counts:


,status,n
0,ok,88


OPENAI successful predictions preview:


,prompt_variant,row_id,pred_unlearning,pred_target_type,pred_agency,confidence,rationale,input_tokens,output_tokens,reasoning_tokens,cost_usd
0,direct_no_context,14:2,Yes,misc_organizational,FEMA,0.66,The paragraph raises whether FEMA should be di...,385,192,127,0.000317
1,direct_no_context,14:3,No,none,FEMA,0.82,The paragraph describes GAO reviews and policy...,640,267,205,0.000462
2,direct_no_context,14:4,Yes,misc_organizational,FEMA,0.62,The paragraph raises concerns about FEMA’s org...,400,171,97,0.000294
3,direct_no_context,14:5,Yes,capabilities,The White House,0.62,The text argues against relying on the traditi...,443,228,166,0.000374
4,direct_no_context,14:6,No,none,NaN,0.84,The paragraph discusses improving disaster res...,436,87,31,0.000196
5,direct_no_context,14:7,Yes,laws_plans_policies,NaN,0.78,It calls for replacing unquestioned adherence ...,444,114,49,0.000231
6,direct_no_context,14:8,No,none,null,0.78,The paragraph discusses planning and recommend...,523,165,103,0.000311
7,direct_no_context,14:9,No,none,Department of Defense (DOD),0.62,The paragraph describes assessment capability ...,425,202,132,0.000338
8,direct_no_context,14:12,No,none,NaN,0.74,The paragraph argues that organizational place...,853,176,114,0.000391
9,direct_no_context,14:13,No,none,NaN,0.86,The paragraph describes damage and rebuilding ...,416,83,26,0.000187


In [ ]:
#5.4
OPENAI_CFG = get_provider_config("openai")
openai_preflight_df = estimate_full_grid(test_df, [OPENAI_CFG], PROMPT_VARIANTS_TO_RUN)
openai_estimated_cost = openai_preflight_df["estimated_cost_usd"].sum()
print("OpenAI preflight estimated cost: $", round(openai_estimated_cost, 6))
display(openai_preflight_df)

if RUN_OPENAI_CALLS and not os.getenv(OPENAI_CFG["api_key_env"]):
    raise RuntimeError(f"{OPENAI_CFG['api_key_env']} is not set.")
if RUN_OPENAI_CALLS and openai_estimated_cost > MAX_ESTIMATED_COST_USD_PER_PROVIDER:
    raise RuntimeError(
        f"OpenAI estimated cost ${openai_estimated_cost:.4f} exceeds per-provider guard "
        f"${MAX_ESTIMATED_COST_USD_PER_PROVIDER:.2f}."
    )

OPENAI_RESULTS_JSONL = provider_results_jsonl("openai")
results_openai_df = run_single_provider(
    test_df,
    OPENAI_CFG,
    PROMPT_VARIANTS_TO_RUN,
    OPENAI_RESULTS_JSONL,
    RUN_OPENAI_CALLS,
)
print("OpenAI snapshot:", provider_predictions_csv("openai"))
provider_diagnostics(results_openai_df, "openai")


OpenAI preflight estimated cost: $ 0.027698


,provider,model,model_effort,prompt_version,prompt_variant,n_rows,estimated_input_tokens,assumed_output_tokens,estimated_total_tokens,estimated_cost_usd
0,openai,gpt-5.4,reasoning_effort=low,v4_provider_separated_confidence_api_fixes,direct_no_context,11,4133,1100,5233,0.002202
1,openai,gpt-5.4,reasoning_effort=low,v4_provider_separated_confidence_api_fixes,definitions_only,11,25847,1100,26947,0.006544
2,openai,gpt-5.4,reasoning_effort=low,v4_provider_separated_confidence_api_fixes,definitions_examples_no_metadata,11,39355,1100,40455,0.009246
3,openai,gpt-5.4,reasoning_effort=low,v4_provider_separated_confidence_api_fixes,definitions_examples_with_metadata,11,41654,1100,42754,0.009706


Provider: openai / gpt-5.4
Resume log: outputs_unlearning_ab_test_v4/openai_raw_results_v4_provider_separated_confidence_api_fixes.jsonl
Prior unique run keys: 88; successful resume keys: 88


openai A/B:   0%|          | 0/44 [00:00<?, ?it/s]

OpenAI snapshot: outputs_unlearning_ab_test_v4/openai_predictions_v4_provider_separated_confidence_api_fixes.csv

OPENAI status counts:


,status,n
0,ok,132


OPENAI successful predictions preview:


,prompt_variant,row_id,pred_unlearning,pred_target_type,pred_agency,confidence,rationale,input_tokens,output_tokens,reasoning_tokens,cost_usd
0,direct_no_context,14:2,Yes,misc_organizational,FEMA,0.66,The paragraph raises whether FEMA should be di...,385,192,127,0.000317
1,direct_no_context,14:3,No,none,FEMA,0.82,The paragraph describes GAO reviews and policy...,640,267,205,0.000462
2,direct_no_context,14:4,Yes,misc_organizational,FEMA,0.62,The paragraph raises concerns about FEMA’s org...,400,171,97,0.000294
3,direct_no_context,14:5,Yes,capabilities,The White House,0.62,The text argues against relying on the traditi...,443,228,166,0.000374
4,direct_no_context,14:6,No,none,NaN,0.84,The paragraph discusses improving disaster res...,436,87,31,0.000196
5,direct_no_context,14:7,Yes,laws_plans_policies,NaN,0.78,It calls for replacing unquestioned adherence ...,444,114,49,0.000231
6,direct_no_context,14:8,No,none,null,0.78,The paragraph discusses planning and recommend...,523,165,103,0.000311
7,direct_no_context,14:9,No,none,Department of Defense (DOD),0.62,The paragraph describes assessment capability ...,425,202,132,0.000338
8,direct_no_context,14:12,No,none,NaN,0.74,The paragraph argues that organizational place...,853,176,114,0.000391
9,direct_no_context,14:13,No,none,NaN,0.86,The paragraph describes damage and rebuilding ...,416,83,26,0.000187


In [ ]:
results_openai_df3 = results_openai_df[results_openai_df['model'] == 'gpt-5.4']

In [ ]:
results_openai_df3[['prompt_variant', 'pred_unlearning']]

,prompt_variant,pred_unlearning
88,direct_no_context,Yes
89,direct_no_context,Yes
90,direct_no_context,No
91,direct_no_context,Yes
92,direct_no_context,Yes
93,direct_no_context,Yes
94,direct_no_context,Yes
95,direct_no_context,No
96,direct_no_context,Yes
97,direct_no_context,Yes


### 8B. Run Anthropic / Claude Sonnet 5 only

This cell calls only Anthropic. The Sonnet 5 request intentionally omits `temperature`; it uses `messages.parse()` with the shared Pydantic schema and `output_config={"effort": "medium"}`.


In [ ]:
# Sonnet 5.0 Low
ANTHROPIC_CFG = get_provider_config("anthropic")
anthropic_preflight_df = estimate_full_grid(test_df, [ANTHROPIC_CFG], PROMPT_VARIANTS_TO_RUN)
anthropic_estimated_cost = anthropic_preflight_df["estimated_cost_usd"].sum()
print("Anthropic preflight estimated cost: $", round(anthropic_estimated_cost, 6))
display(anthropic_preflight_df)

if RUN_ANTHROPIC_CALLS and not os.getenv(ANTHROPIC_CFG["api_key_env"]):
    raise RuntimeError(f"{ANTHROPIC_CFG['api_key_env']} is not set.")
if RUN_ANTHROPIC_CALLS and anthropic_estimated_cost > MAX_ESTIMATED_COST_USD_PER_PROVIDER:
    raise RuntimeError(
        f"Anthropic estimated cost ${anthropic_estimated_cost:.4f} exceeds per-provider guard "
        f"${MAX_ESTIMATED_COST_USD_PER_PROVIDER:.2f}."
    )

ANTHROPIC_RESULTS_JSONL = provider_results_jsonl("anthropic")
results_anthropic_df = run_single_provider(
    test_df,
    ANTHROPIC_CFG,
    PROMPT_VARIANTS_TO_RUN,
    ANTHROPIC_RESULTS_JSONL,
    RUN_ANTHROPIC_CALLS,
)
print("Anthropic snapshot:", provider_predictions_csv("anthropic"))
provider_diagnostics(results_anthropic_df, "anthropic")


Anthropic preflight estimated cost: $ 0.336826


,provider,model,model_effort,prompt_version,prompt_variant,n_rows,estimated_input_tokens,assumed_output_tokens,estimated_total_tokens,estimated_cost_usd
0,anthropic,claude-sonnet-5,effort=low,v4_provider_separated_confidence_api_fixes,direct_no_context,11,5164,1100,6264,0.021328
1,anthropic,claude-sonnet-5,effort=low,v4_provider_separated_confidence_api_fixes,definitions_only,11,34567,1100,35667,0.080134
2,anthropic,claude-sonnet-5,effort=low,v4_provider_separated_confidence_api_fixes,definitions_examples_no_metadata,11,52377,1100,53477,0.115754
3,anthropic,claude-sonnet-5,effort=low,v4_provider_separated_confidence_api_fixes,definitions_examples_with_metadata,11,54305,1100,55405,0.119610


Provider: anthropic / claude-sonnet-5
Resume log: outputs_unlearning_ab_test_v4/anthropic_raw_results_v4_provider_separated_confidence_api_fixes.jsonl
Prior unique run keys: 0; successful resume keys: 0


anthropic A/B:   0%|          | 0/44 [00:00<?, ?it/s]

Anthropic snapshot: outputs_unlearning_ab_test_v4/anthropic_predictions_v4_provider_separated_confidence_api_fixes.csv

ANTHROPIC status counts:


,status,n
0,ok,44


ANTHROPIC successful predictions preview:


,prompt_variant,row_id,pred_unlearning,pred_target_type,pred_agency,confidence,rationale,input_tokens,output_tokens,reasoning_tokens,cost_usd
0,direct_no_context,14:2,Yes,misc_organizational,FEMA,0.65,Discusses potentially disbanding FEMA or movin...,916,83,0,0.002662
1,direct_no_context,14:3,No,none,NaN,0.85,Describes past GAO reviews and recommendations...,1368,71,0,0.003446
2,direct_no_context,14:4,Yes,misc_organizational,FEMA,0.60,Discusses potentially disbanding FEMA or movin...,939,87,0,0.002748
3,direct_no_context,14:5,No,none,NaN,0.80,Discusses scope of catastrophic incident polic...,1024,82,0,0.002868
4,direct_no_context,14:6,No,none,NaN,0.85,The paragraph discusses recommendations to imp...,1012,87,0,0.002894
5,direct_no_context,14:7,Yes,laws_plans_policies,NaN,0.75,Calls for replacing long-standing policies and...,1018,85,0,0.002886
6,direct_no_context,14:8,No,none,NaN,0.80,Describes ongoing capability gaps and recommen...,1155,73,0,0.003040
7,direct_no_context,14:9,No,none,NaN,0.70,Describes a capability gap and lack of advance...,991,70,0,0.002682
8,direct_no_context,14:12,Yes,capabilities,FEMA,0.65,Describes loss of experienced staff ('FEMA bra...,1754,97,0,0.004478
9,direct_no_context,14:13,No,none,NaN,0.75,Describes damage to health infrastructure and ...,969,83,0,0.002768


In [ ]:
results_anthropic_df[['prompt_variant', 'pred_unlearning']]

,prompt_variant,pred_unlearning
0,direct_no_context,Yes
1,direct_no_context,No
2,direct_no_context,Yes
3,direct_no_context,No
4,direct_no_context,No
5,direct_no_context,Yes
6,direct_no_context,No
7,direct_no_context,No
8,direct_no_context,Yes
9,direct_no_context,No


In [ ]:
#Haiku 4.5 low
ANTHROPIC_CFG = get_provider_config("anthropic")
anthropic_preflight_df = estimate_full_grid(test_df, [ANTHROPIC_CFG], PROMPT_VARIANTS_TO_RUN)
anthropic_estimated_cost = anthropic_preflight_df["estimated_cost_usd"].sum()
print("Anthropic preflight estimated cost: $", round(anthropic_estimated_cost, 6))
display(anthropic_preflight_df)

if RUN_ANTHROPIC_CALLS and not os.getenv(ANTHROPIC_CFG["api_key_env"]):
    raise RuntimeError(f"{ANTHROPIC_CFG['api_key_env']} is not set.")
if RUN_ANTHROPIC_CALLS and anthropic_estimated_cost > MAX_ESTIMATED_COST_USD_PER_PROVIDER:
    raise RuntimeError(
        f"Anthropic estimated cost ${anthropic_estimated_cost:.4f} exceeds per-provider guard "
        f"${MAX_ESTIMATED_COST_USD_PER_PROVIDER:.2f}."
    )

ANTHROPIC_RESULTS_JSONL = provider_results_jsonl("anthropic")
results_anthropic_df = run_single_provider(
    test_df,
    ANTHROPIC_CFG,
    PROMPT_VARIANTS_TO_RUN,
    ANTHROPIC_RESULTS_JSONL,
    RUN_ANTHROPIC_CALLS,
)
print("Anthropic snapshot:", provider_predictions_csv("anthropic"))
provider_diagnostics(results_anthropic_df, "anthropic")


Anthropic preflight estimated cost: $ 0.168413


,provider,model,model_effort,prompt_version,prompt_variant,n_rows,estimated_input_tokens,assumed_output_tokens,estimated_total_tokens,estimated_cost_usd
0,anthropic,claude-haiku-4-5-20251001,effort=omitted,v4_provider_separated_confidence_api_fixes,direct_no_context,11,5164,1100,6264,0.010664
1,anthropic,claude-haiku-4-5-20251001,effort=omitted,v4_provider_separated_confidence_api_fixes,definitions_only,11,34567,1100,35667,0.040067
2,anthropic,claude-haiku-4-5-20251001,effort=omitted,v4_provider_separated_confidence_api_fixes,definitions_examples_no_metadata,11,52377,1100,53477,0.057877
3,anthropic,claude-haiku-4-5-20251001,effort=omitted,v4_provider_separated_confidence_api_fixes,definitions_examples_with_metadata,11,54305,1100,55405,0.059805


Provider: anthropic / claude-haiku-4-5-20251001
Resume log: outputs_unlearning_ab_test_v4/anthropic_raw_results_v4_provider_separated_confidence_api_fixes.jsonl
Prior unique run keys: 45; successful resume keys: 44


anthropic A/B:   0%|          | 0/44 [00:00<?, ?it/s]

Anthropic snapshot: outputs_unlearning_ab_test_v4/anthropic_predictions_v4_provider_separated_confidence_api_fixes.csv

ANTHROPIC status counts:


,status,n
0,ok,88
1,error,1


ANTHROPIC failures / parse failures:


,prompt_variant,row_id,attempt_number,status,parse_error,raw_response
44,direct_no_context,14:2,1,error,NotFoundError: Error code: 404 - {'type': 'err...,


ANTHROPIC successful predictions preview:


,prompt_variant,row_id,pred_unlearning,pred_target_type,pred_agency,confidence,rationale,input_tokens,output_tokens,reasoning_tokens,cost_usd
0,direct_no_context,14:2,Yes,misc_organizational,FEMA,0.65,Discusses potentially disbanding FEMA or movin...,916,83,0,0.002662
1,direct_no_context,14:3,No,none,NaN,0.85,Describes past GAO reviews and recommendations...,1368,71,0,0.003446
2,direct_no_context,14:4,Yes,misc_organizational,FEMA,0.60,Discusses potentially disbanding FEMA or movin...,939,87,0,0.002748
3,direct_no_context,14:5,No,none,NaN,0.80,Discusses scope of catastrophic incident polic...,1024,82,0,0.002868
4,direct_no_context,14:6,No,none,NaN,0.85,The paragraph discusses recommendations to imp...,1012,87,0,0.002894
5,direct_no_context,14:7,Yes,laws_plans_policies,NaN,0.75,Calls for replacing long-standing policies and...,1018,85,0,0.002886
6,direct_no_context,14:8,No,none,NaN,0.80,Describes ongoing capability gaps and recommen...,1155,73,0,0.003040
7,direct_no_context,14:9,No,none,NaN,0.70,Describes a capability gap and lack of advance...,991,70,0,0.002682
8,direct_no_context,14:12,Yes,capabilities,FEMA,0.65,Describes loss of experienced staff ('FEMA bra...,1754,97,0,0.004478
9,direct_no_context,14:13,No,none,NaN,0.75,Describes damage to health infrastructure and ...,969,83,0,0.002768


In [ ]:
results_anthropic_df2 = results_anthropic_df[results_anthropic_df['model'] == 'claude-haiku-4-5-20251001']

In [ ]:
results_anthropic_df2[['prompt_variant', 'pred_unlearning']]

,prompt_variant,pred_unlearning
45,direct_no_context,No
46,direct_no_context,No
47,direct_no_context,Yes
48,direct_no_context,No
49,direct_no_context,No
50,direct_no_context,Yes
51,direct_no_context,No
52,direct_no_context,Yes
53,direct_no_context,No
54,direct_no_context,No


### 8C. Run Google / Gemini 3.5 Flash only

This cell calls only Gemini. It uses the still-supported `generateContent` API because the project will later benefit from Gemini Batch API, and sends native JSON Schema through `response_json_schema`. The explicit Gemini 2.5 `thinking_budget` is recorded in the run key.


In [ ]:
GEMINI_CFG = get_provider_config("gemini")
gemini_preflight_df = estimate_full_grid(test_df, [GEMINI_CFG], PROMPT_VARIANTS_TO_RUN)
gemini_estimated_cost = gemini_preflight_df["estimated_cost_usd"].sum()
print("Gemini preflight estimated cost: $", round(gemini_estimated_cost, 6))
display(gemini_preflight_df)

if RUN_GEMINI_CALLS and not os.getenv(GEMINI_CFG["api_key_env"]):
    raise RuntimeError(f"{GEMINI_CFG['api_key_env']} is not set.")
if RUN_GEMINI_CALLS and gemini_estimated_cost > MAX_ESTIMATED_COST_USD_PER_PROVIDER:
    raise RuntimeError(
        f"Gemini estimated cost ${gemini_estimated_cost:.4f} exceeds per-provider guard "
        f"${MAX_ESTIMATED_COST_USD_PER_PROVIDER:.2f}."
    )

GEMINI_RESULTS_JSONL = provider_results_jsonl("gemini")
results_gemini_df = run_single_provider(
    test_df,
    GEMINI_CFG,
    PROMPT_VARIANTS_TO_RUN,
    GEMINI_RESULTS_JSONL,
    RUN_GEMINI_CALLS,
)
print("Gemini snapshot:", provider_predictions_csv("gemini"))
provider_diagnostics(results_gemini_df, "gemini")


Gemini preflight estimated cost: $ 0.25922


,provider,model,model_effort,prompt_version,prompt_variant,n_rows,estimated_input_tokens,assumed_output_tokens,estimated_total_tokens,estimated_cost_usd
0,gemini,gemini-3.5-flash,thinking_level=low,v4_provider_separated_confidence_api_fixes,direct_no_context,11,5164,1100,6264,0.017646
1,gemini,gemini-3.5-flash,thinking_level=low,v4_provider_separated_confidence_api_fixes,definitions_only,11,34567,1100,35667,0.061750
2,gemini,gemini-3.5-flash,thinking_level=low,v4_provider_separated_confidence_api_fixes,definitions_examples_no_metadata,11,52377,1100,53477,0.088466
3,gemini,gemini-3.5-flash,thinking_level=low,v4_provider_separated_confidence_api_fixes,definitions_examples_with_metadata,11,54305,1100,55405,0.091358


Provider: gemini / gemini-3.5-flash
Resume log: outputs_unlearning_ab_test_v4/gemini_raw_results_v4_provider_separated_confidence_api_fixes.jsonl
Prior unique run keys: 42; successful resume keys: 39


gemini A/B:   0%|          | 0/44 [00:00<?, ?it/s]


Gemini daily free-tier quota exhausted.
The same request will be retried after the daily quota reset.
Resume time: 2026-07-09 00:01:30 PDT
Sleeping for 8.67 hours...


KeyboardInterrupt: 

In [ ]:
results_gemini_df

,run_key,attempt_number,timestamp_utc,provider,model,model_effort,prompt_version,prompt_sha256,prompt_variant,row_id,...,pred_agency,confidence,rationale,input_tokens,output_tokens,reasoning_tokens,total_tokens,cost_usd,latency_seconds,parse_error
0,4d3eb6b953d3b5fedaa6c8de1695d5d88c39a238d3de27...,1,2026-07-08T21:07:09.385758+00:00,gemini,gemini-2.5-flash,thinking_budget=1024,v4_provider_separated_confidence_api_fixes,dfa06a441ee3c990d61ef433434962ffd422a900dc680a...,direct_no_context,14:2,...,FEMA,0.90,The paragraph discusses re-evaluating and pote...,262,61,328,651,0.000231,2.902178,
1,e4b3e05f23e5eee068a0f51fedf1671462859e35921454...,1,2026-07-08T21:07:15.481052+00:00,gemini,gemini-2.5-flash,thinking_budget=1024,v4_provider_separated_confidence_api_fixes,880f5d104fe26344be02a80b044c6ae15179647e6f5464...,direct_no_context,14:3,...,,NaN,,0,0,0,0,NaN,0.000000,APIErrorForRetry: Gemini returned JSON that fa...
2,7127db62a09186e1a74dd346ca1844b8481c04f5c9292b...,1,2026-07-08T21:28:30.025782+00:00,gemini,gemini-2.5-flash,thinking_budget=512,v4_provider_separated_confidence_api_fixes,dfa06a441ee3c990d61ef433434962ffd422a900dc680a...,direct_no_context,14:2,...,FEMA,0.95,The paragraph discusses whether FEMA should be...,262,64,220,546,0.000239,4.084862,
3,9b61d9cf7c9429dd3595759736401c9117e7ace14fcfef...,1,2026-07-08T21:28:34.460155+00:00,gemini,gemini-2.5-flash,thinking_budget=512,v4_provider_separated_confidence_api_fixes,880f5d104fe26344be02a80b044c6ae15179647e6f5464...,direct_no_context,14:3,...,FEMA,0.90,The paragraph describes recommendations to cha...,520,65,393,978,0.000318,2.948437,
4,76a65fd5857d18c9f1b15aeb0aef5eb4b60ca2177a2cae...,1,2026-07-08T21:28:37.752992+00:00,gemini,gemini-2.5-flash,thinking_budget=512,v4_provider_separated_confidence_api_fixes,d493feec8f0083c8b3a2cece347956f57f211b5958e49a...,direct_no_context,14:4,...,FEMA,0.90,The paragraph discusses critically evaluating ...,275,59,375,709,0.000230,2.755560,
5,7f6454245b6c691fadba0cdc2377cdebc4fe7678b9d070...,1,2026-07-08T21:28:40.966159+00:00,gemini,gemini-2.5-flash,thinking_budget=512,v4_provider_separated_confidence_api_fixes,e80bcbd4b2573d425905722925aff49f3bc631f945fc26...,direct_no_context,14:5,...,The White House,0.90,The paragraph discusses moving away from the t...,322,69,389,780,0.000269,2.936249,
6,4ab0311b71376255377b9e227ed472cf3a2c1ec73a57e9...,1,2026-07-08T21:28:44.258789+00:00,gemini,gemini-2.5-flash,thinking_budget=512,v4_provider_separated_confidence_api_fixes,d0e6a895bb0858a5f27ebc87b198e6b06a233ea3ccfbd4...,direct_no_context,14:6,...,FEMA,0.90,The paragraph reinforces existing recommendati...,314,53,378,745,0.000227,2.847262,
7,1aa9805ff1cd55b0d93e5673c01a23a553825c1f2aaf8e...,1,2026-07-08T21:28:47.445902+00:00,gemini,gemini-2.5-flash,thinking_budget=512,v4_provider_separated_confidence_api_fixes,1117cbe6162069cbea56bfb68094193018f596b34df489...,direct_no_context,14:7,...,NaN,0.90,The text explicitly calls for replacing long-s...,320,65,304,689,0.000258,2.477996,
8,6c1c6bcf568d204a89180a5af79d05be6d6999a31e6b23...,1,2026-07-08T21:28:50.267503+00:00,gemini,gemini-2.5-flash,thinking_budget=512,v4_provider_separated_confidence_api_fixes,40391b039f84ae5d63acbf33230861069844f96f7da5d4...,direct_no_context,14:8,...,Department of Homeland Security,0.90,The paragraph identifies existing problems and...,403,65,381,849,0.000283,3.064763,
9,13936431827ce914ab4ea4c004601b39aa508798b4ac37...,1,2026-07-08T21:30:03.556729+00:00,gemini,gemini-2.5-flash,thinking_budget=512,v4_provider_separated_confidence_api_fixes,d7ec582b2dd894e93ecdd7db96132a19f8e1915a0d6bd9...,direct_no_context,14:9,...,DOD,0.90,The paragraph highlights a past failure in adv...,300,54,427,781,0.000225,3.021783,


In [ ]:
results_gemini_df2 = results_gemini_df[results_gemini_df['model'] == 'gemini-3.5-flash']

In [ ]:
results_gemini_df2[['prompt_variant', 'pred_unlearning']]

,prompt_variant,pred_unlearning
19,direct_no_context,Yes
20,direct_no_context,Yes
21,direct_no_context,Yes
22,direct_no_context,Yes
23,direct_no_context,Yes
24,direct_no_context,Yes
25,direct_no_context,Yes
26,direct_no_context,No
27,direct_no_context,Yes
28,direct_no_context,Yes


## 9. Combine whichever provider runs exist

This cell does not call any API. It combines the in-memory provider result DataFrames that have been created in this notebook session. Provider CSV/JSONL snapshots remain separate for diagnosis.


In [ ]:
provider_frames = []
for variable_name in ["results_openai_df", "results_anthropic_df", "results_gemini_df"]:
    frame = globals().get(variable_name)
    if isinstance(frame, pd.DataFrame) and not frame.empty:
        provider_frames.append(frame.copy())

if not provider_frames:
    raise RuntimeError("No provider results are available. Run at least one provider section first.")

results_df = pd.concat(provider_frames, ignore_index=True)
print("Combined result shape:", results_df.shape)
display(results_df.groupby(["provider", "status"], as_index=False).size())


Combined result shape: (263, 28)


,provider,status,size
0,anthropic,error,1
1,anthropic,ok,88
2,gemini,error,3
3,gemini,ok,39
4,openai,ok,132


### Score interpretation

Because the current GPT Test rows are all `Unlearning = Yes`, the binary score below is **Yes recall** for this benchmark. Add human-coded `No` rows before using overall accuracy, precision, or F1 to select a production model.


## 10. Evaluate predictions and summarize cost

In [ ]:
def label_to_binary(series: pd.Series) -> pd.Series:
    return series.apply(
        lambda x: 1 if normalize_yes_no(x) == "Yes" else (0 if normalize_yes_no(x) == "No" else np.nan)
    )


def summarize_metrics(df: pd.DataFrame) -> pd.DataFrame:
    scored = df[df["status"].eq("ok")].copy()
    scored["y_true"] = label_to_binary(scored["ground_truth_unlearning"])
    scored["y_pred"] = label_to_binary(scored["pred_unlearning"])
    scored = scored.dropna(subset=["y_true", "y_pred"])

    rows = []
    group_cols = ["provider", "model", "model_effort", "prompt_variant"]

    for keys, g in scored.groupby(group_cols):
        provider, model, model_effort, variant = keys
        y_true = g["y_true"].astype(int)
        y_pred = g["y_pred"].astype(int)

        tp = int(((y_true == 1) & (y_pred == 1)).sum())
        tn = int(((y_true == 0) & (y_pred == 0)).sum())
        fp = int(((y_true == 0) & (y_pred == 1)).sum())
        fn = int(((y_true == 1) & (y_pred == 0)).sum())

        accuracy = (tp + tn) / len(g) if len(g) else np.nan
        precision_yes = tp / (tp + fp) if (tp + fp) else np.nan
        recall_yes = tp / (tp + fn) if (tp + fn) else np.nan
        f1_yes = (
            2 * precision_yes * recall_yes / (precision_yes + recall_yes)
            if pd.notna(precision_yes) and pd.notna(recall_yes) and (precision_yes + recall_yes)
            else np.nan
        )

        rows.append({
            "provider": provider,
            "model": model,
            "model_effort": model_effort,
            "prompt_variant": variant,
            "n_scored": len(g),
            "accuracy": accuracy,
            "precision_yes": precision_yes,
            "recall_yes": recall_yes,
            "f1_yes": f1_yes,
            "tp": tp,
            "tn": tn,
            "fp": fp,
            "fn": fn,
            "mean_confidence": g["confidence"].mean(),
            "input_tokens": g["input_tokens"].sum(),
            "output_tokens": g["output_tokens"].sum(),
            "reasoning_tokens": g["reasoning_tokens"].sum(),
            "total_tokens": g["total_tokens"].sum(),
            "cost_usd": g["cost_usd"].sum(),
            "cost_per_scored_row": g["cost_usd"].sum() / len(g) if len(g) else np.nan,
            "cost_per_correct_binary_label": g["cost_usd"].sum() / (tp + tn) if (tp + tn) else np.nan,
            "avg_latency_seconds": g["latency_seconds"].mean(),
        })

    if not rows:
        return pd.DataFrame()

    return pd.DataFrame(rows).sort_values(
        ["recall_yes", "cost_per_correct_binary_label", "cost_usd"],
        ascending=[False, True, True],
    )


correct_series = (
    results_df["ground_truth_unlearning"].apply(normalize_yes_no)
    == results_df["pred_unlearning"].apply(normalize_yes_no)
)
results_df["correct"] = correct_series.astype("object")
results_df.loc[results_df["status"] != "ok", "correct"] = np.nan

metrics_df = summarize_metrics(results_df)

usage_by_variant_df = results_df.groupby(
    ["provider", "model", "model_effort", "prompt_variant", "status"],
    as_index=False,
).agg(
    n_rows=("row_id", "count"),
    input_tokens=("input_tokens", "sum"),
    output_tokens=("output_tokens", "sum"),
    reasoning_tokens=("reasoning_tokens", "sum"),
    total_tokens=("total_tokens", "sum"),
    cost_usd=("cost_usd", "sum"),
    avg_confidence=("confidence", "mean"),
    avg_latency_seconds=("latency_seconds", "mean"),
)

usage_by_model_df = results_df.groupby(
    ["provider", "model", "model_effort"],
    as_index=False,
).agg(
    n_records=("row_id", "count"),
    input_tokens=("input_tokens", "sum"),
    output_tokens=("output_tokens", "sum"),
    reasoning_tokens=("reasoning_tokens", "sum"),
    total_tokens=("total_tokens", "sum"),
    cost_usd=("cost_usd", "sum"),
    avg_confidence=("confidence", "mean"),
)

print("Metrics by model/prompt:")
display(metrics_df)
print("Usage by variant:")
display(usage_by_variant_df)
print("Total usage by model:")
display(usage_by_model_df)


Metrics by model/prompt:


,provider,model,model_effort,prompt_variant,n_scored,accuracy,precision_yes,recall_yes,f1_yes,tp,...,fn,mean_confidence,input_tokens,output_tokens,reasoning_tokens,total_tokens,cost_usd,cost_per_scored_row,cost_per_correct_binary_label,avg_latency_seconds
8,gemini,gemini-2.5-flash,thinking_budget=1024,direct_no_context,1,1.000000,1.0,1.000000,1.000000,1,...,0,0.900000,262,61,328,651,0.000231,0.000231,0.000231,2.902178
0,anthropic,claude-haiku-4-5-20251001,effort=omitted,definitions_examples_no_metadata,11,1.000000,1.0,1.000000,1.000000,11,...,0,0.830000,49335,936,0,50271,0.054015,0.004910,0.004910,1.639208
6,anthropic,claude-sonnet-5,effort=low,definitions_only,11,1.000000,1.0,1.000000,1.000000,11,...,0,0.677273,50707,1117,0,51824,0.112584,0.010235,0.010235,2.503172
4,anthropic,claude-sonnet-5,effort=low,definitions_examples_no_metadata,11,1.000000,1.0,1.000000,1.000000,11,...,0,0.709091,73642,1121,0,74763,0.158494,0.014409,0.014409,3.080161
5,anthropic,claude-sonnet-5,effort=low,definitions_examples_with_metadata,11,1.000000,1.0,1.000000,1.000000,11,...,0,0.704545,77096,1051,0,78147,0.164702,0.014973,0.014973,2.391853
12,gemini,gemini-3.5-flash,thinking_level=low,direct_no_context,11,0.909091,1.0,0.909091,0.952381,10,...,1,0.859091,4159,847,2276,7282,0.013861,0.001260,0.001386,7.005746
2,anthropic,claude-haiku-4-5-20251001,effort=omitted,definitions_only,11,0.909091,1.0,0.909091,0.952381,10,...,1,0.847273,34155,985,0,35140,0.039080,0.003553,0.003908,1.522081
10,gemini,gemini-2.5-flash,thinking_budget=512,direct_no_context,11,0.818182,1.0,0.818182,0.900000,9,...,2,0.904545,4159,694,3981,8834,0.002983,0.000271,0.000331,2.951038
16,openai,gpt-5.4,reasoning_effort=low,direct_no_context,11,0.818182,1.0,0.818182,0.900000,9,...,2,0.815455,5497,1679,1029,7176,0.003198,0.000291,0.000355,2.309942
13,openai,gpt-5.4,reasoning_effort=low,definitions_examples_no_metadata,11,0.818182,1.0,0.818182,0.900000,9,...,2,0.862727,40598,1599,938,42197,0.010118,0.000920,0.001124,2.143202


Usage by variant:


,provider,model,model_effort,prompt_variant,status,n_rows,input_tokens,output_tokens,reasoning_tokens,total_tokens,cost_usd,avg_confidence,avg_latency_seconds
0,anthropic,claude-haiku-4-5-20251001,effort=omitted,definitions_examples_no_metadata,ok,11,49335,936,0,50271,0.054015,0.830000,1.639208
1,anthropic,claude-haiku-4-5-20251001,effort=omitted,definitions_examples_with_metadata,ok,11,51931,934,0,52865,0.056601,0.808182,1.697688
2,anthropic,claude-haiku-4-5-20251001,effort=omitted,definitions_only,ok,11,34155,985,0,35140,0.039080,0.847273,1.522081
3,anthropic,claude-haiku-4-5-20251001,effort=omitted,direct_no_context,ok,11,9119,818,0,9937,0.013209,0.922727,1.788631
4,anthropic,claude-haiku-4.5,effort=low,direct_no_context,error,1,0,0,0,0,0.000000,NaN,0.000000
5,anthropic,claude-sonnet-5,effort=low,definitions_examples_no_metadata,ok,11,73642,1121,0,74763,0.158494,0.709091,3.080161
6,anthropic,claude-sonnet-5,effort=low,definitions_examples_with_metadata,ok,11,77096,1051,0,78147,0.164702,0.704545,2.391853
7,anthropic,claude-sonnet-5,effort=low,definitions_only,ok,11,50707,1117,0,51824,0.112584,0.677273,2.503172
8,anthropic,claude-sonnet-5,effort=low,direct_no_context,ok,11,12284,893,0,13177,0.033498,0.745455,2.202107
9,gemini,gemini-2.5-flash,thinking_budget=1024,direct_no_context,error,1,0,0,0,0,0.000000,NaN,0.000000


Total usage by model:


,provider,model,model_effort,n_records,input_tokens,output_tokens,reasoning_tokens,total_tokens,cost_usd,avg_confidence
0,anthropic,claude-haiku-4-5-20251001,effort=omitted,44,144540,3673,0,148213,0.162905,0.852045
1,anthropic,claude-haiku-4.5,effort=low,1,0,0,0,0,0.000000,NaN
2,anthropic,claude-sonnet-5,effort=low,44,213729,4182,0,217911,0.469278,0.709091
3,gemini,gemini-2.5-flash,thinking_budget=1024,2,262,61,328,651,0.000231,0.900000
4,gemini,gemini-2.5-flash,thinking_budget=512,17,15817,1034,5915,22766,0.007330,0.906250
5,gemini,gemini-3.5-flash,thinking_level=low,23,30241,1702,4921,36864,0.060679,0.884091
6,openai,gpt-5.4,reasoning_effort=low,44,116148,6557,3897,122705,0.031426,0.854773
7,openai,gpt-5.4-mini,reasoning_effort=low,44,116148,5861,3234,122009,0.030556,0.907500
8,openai,gpt-5.4-nano,reasoning_effort=medium,44,116148,9770,6713,125918,0.035442,0.750455


## 11. Save combined outputs

In [ ]:
PREDICTIONS_CSV = OUTPUT_DIR / f"predictions_long_{PROMPT_VERSION}.csv"
METRICS_CSV = OUTPUT_DIR / f"metrics_by_model_prompt_{PROMPT_VERSION}.csv"
USAGE_VARIANT_CSV = OUTPUT_DIR / f"usage_by_variant_{PROMPT_VERSION}.csv"
USAGE_MODEL_CSV = OUTPUT_DIR / f"usage_by_model_{PROMPT_VERSION}.csv"
PROMPT_AUDIT_CSV = OUTPUT_DIR / f"prompt_audit_{PROMPT_VERSION}.csv"
OUTPUT_XLSX = OUTPUT_DIR / f"unlearning_ab_test_results_{PROMPT_VERSION}.xlsx"

results_df.to_csv(PREDICTIONS_CSV, index=False)
metrics_df.to_csv(METRICS_CSV, index=False)
usage_by_variant_df.to_csv(USAGE_VARIANT_CSV, index=False)
usage_by_model_df.to_csv(USAGE_MODEL_CSV, index=False)
prompt_audit_df.to_csv(PROMPT_AUDIT_CSV, index=False)

estimate_df = estimate_full_grid(
    test_df,
    active_model_configs(require_api_key=False),
    PROMPT_VARIANTS_TO_RUN,
)

with pd.ExcelWriter(OUTPUT_XLSX, engine="openpyxl") as writer:
    results_df.to_excel(writer, index=False, sheet_name="Predictions_Long")
    metrics_df.to_excel(writer, index=False, sheet_name="Metrics")
    usage_by_variant_df.to_excel(writer, index=False, sheet_name="Usage_By_Variant")
    usage_by_model_df.to_excel(writer, index=False, sheet_name="Usage_By_Model")
    estimate_df.to_excel(writer, index=False, sheet_name="Preflight_Estimates")
    prompt_audit_df.to_excel(writer, index=False, sheet_name="Prompt_Audit")

print("Saved combined outputs:")
for path in [
    PREDICTIONS_CSV,
    METRICS_CSV,
    USAGE_VARIANT_CSV,
    USAGE_MODEL_CSV,
    PROMPT_AUDIT_CSV,
    OUTPUT_XLSX,
]:
    print("-", path)

print("\nProvider-specific snapshots:")
for provider in ["openai", "anthropic", "gemini"]:
    print("-", provider_predictions_csv(provider))
    print("-", provider_results_jsonl(provider))


Saved combined outputs:
- outputs_unlearning_ab_test_v4/predictions_long_v4_provider_separated_confidence_api_fixes.csv
- outputs_unlearning_ab_test_v4/metrics_by_model_prompt_v4_provider_separated_confidence_api_fixes.csv
- outputs_unlearning_ab_test_v4/usage_by_variant_v4_provider_separated_confidence_api_fixes.csv
- outputs_unlearning_ab_test_v4/usage_by_model_v4_provider_separated_confidence_api_fixes.csv
- outputs_unlearning_ab_test_v4/prompt_audit_v4_provider_separated_confidence_api_fixes.csv
- outputs_unlearning_ab_test_v4/unlearning_ab_test_results_v4_provider_separated_confidence_api_fixes.xlsx

Provider-specific snapshots:
- outputs_unlearning_ab_test_v4/openai_predictions_v4_provider_separated_confidence_api_fixes.csv
- outputs_unlearning_ab_test_v4/openai_raw_results_v4_provider_separated_confidence_api_fixes.jsonl
- outputs_unlearning_ab_test_v4/anthropic_predictions_v4_provider_separated_confidence_api_fixes.csv
- outputs_unlearning_ab_test_v4/anthropic_raw_results_v4_pr

## 12. Inspect disagreements and provider failures

In [ ]:
if "correct" in results_df.columns:
    disagreements = results_df[
        (results_df["status"] == "ok") & (results_df["correct"] == False)
    ].copy()

    parse_failures = results_df[
        results_df["status"].isin(["parse_failed", "error"])
    ].copy()

    print("Disagreements:", len(disagreements))
    display(disagreements[[
        "provider", "model", "model_effort", "prompt_variant", "row_id",
        "ground_truth_unlearning", "pred_unlearning", "pred_target_type",
        "pred_agency", "confidence", "rationale", "text_content",
    ]].head(50))

    print("Parse/API failures:", len(parse_failures))
    display(parse_failures[[
        "provider", "model", "model_effort", "prompt_variant", "row_id",
        "attempt_number", "status", "parse_error", "raw_response",
    ]].head(50))
else:
    print("No scored results yet. Run at least one provider section.")


Disagreements: 66


,provider,model,model_effort,prompt_variant,row_id,ground_truth_unlearning,pred_unlearning,pred_target_type,pred_agency,confidence,rationale,text_content
1,openai,gpt-5.4-nano,reasoning_effort=medium,direct_no_context,14:3,Yes,No,none,FEMA,0.82,The paragraph describes GAO reviews and policy...,"In addition, we have done a great deal of work..."
4,openai,gpt-5.4-nano,reasoning_effort=medium,direct_no_context,14:6,Yes,No,none,NaN,0.84,The paragraph discusses improving disaster res...,A proactive approach to catastrophic disasters...
6,openai,gpt-5.4-nano,reasoning_effort=medium,direct_no_context,14:8,Yes,No,none,null,0.78,The paragraph discusses planning and recommend...,Our work on interoperable communications ident...
7,openai,gpt-5.4-nano,reasoning_effort=medium,direct_no_context,14:9,Yes,No,none,Department of Defense (DOD),0.62,The paragraph describes assessment capability ...,Damage and needs assessment: Damage and needs ...
8,openai,gpt-5.4-nano,reasoning_effort=medium,direct_no_context,14:12,Yes,No,none,NaN,0.74,The paragraph argues that organizational place...,Because of FEMA’s mission performance during H...
9,openai,gpt-5.4-nano,reasoning_effort=medium,direct_no_context,14:13,Yes,No,none,NaN,0.86,The paragraph describes damage and rebuilding ...,The health care infrastructure in the New Orle...
15,openai,gpt-5.4-nano,reasoning_effort=medium,definitions_only,14:6,Yes,No,none,NaN,0.84,The text recommends proactively improving FEMA...,A proactive approach to catastrophic disasters...
17,openai,gpt-5.4-nano,reasoning_effort=medium,definitions_only,14:8,Yes,No,none,NaN,0.76,The paragraph recommends improved collaborativ...,Our work on interoperable communications ident...
18,openai,gpt-5.4-nano,reasoning_effort=medium,definitions_only,14:9,Yes,No,none,NaN,0.78,The passage identifies a capability/planning g...,Damage and needs assessment: Damage and needs ...
20,openai,gpt-5.4-nano,reasoning_effort=medium,definitions_only,14:13,Yes,No,none,NaN,0.72,The paragraph discusses restoring capacity and...,The health care infrastructure in the New Orle...


Parse/API failures: 4


,provider,model,model_effort,prompt_variant,row_id,attempt_number,status,parse_error,raw_response
176,anthropic,claude-haiku-4.5,effort=low,direct_no_context,14:2,1,error,NotFoundError: Error code: 404 - {'type': 'err...,
222,gemini,gemini-2.5-flash,thinking_budget=1024,direct_no_context,14:3,1,error,APIErrorForRetry: Gemini returned JSON that fa...,
239,gemini,gemini-2.5-flash,thinking_budget=512,definitions_only,14:7,1,error,NonRetryableQuotaError: Daily/free-tier reques...,
262,gemini,gemini-3.5-flash,thinking_level=low,definitions_examples_no_metadata,14:2,1,error,NameError: name 'timedelta' is not defined,


## 13. Interpretation notes

- The **classification task block remains exactly aligned with the GPT-4o-mini notebook**.
- The output schema is now deliberately extended with `confidence`, because the original GPT-4o-mini schema did not request confidence. Without adding this field, no provider can be expected to return or save confidence reliably.
- OpenAI, Claude, and Gemini now use the same semantic schema and prompt content, but provider-native structured-output APIs.
- Keep `STOP_PROVIDER_ON_FIRST_ERROR=True` until all three providers complete at least one successful request. After that, it can be set to `False` if you prefer the run to continue past isolated errors.
- The current 11-row GPT Test set is all positive, so its score is Yes recall rather than a full classifier accuracy benchmark.
